In [20]:
import sys

import pandas as pd
import pyarrow
import sklearn

print(f"pandas version: {pd.__version__}")
print(f"pyarrow version: {pyarrow.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")

pandas version: 3.0.5
pyarrow version: 25.0.1
scikit-learn version: 1.9.0


In [21]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()

print(f"Current directory: {CURRENT_DIR}")
print(f"Current folder name: {CURRENT_DIR.name}")

Current directory: C:\Users\yvett\OneDrive\Documentos\Francisco Fonseca\nfl-pressure-prediction\notebooks
Current folder name: notebooks


In [22]:
# Define centralized project paths
PROJECT_ROOT = CURRENT_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

PFF_PATH = RAW_DATA_DIR / "pffScoutingData.csv"
TEAM_INFO_PATH = RAW_DATA_DIR / "team_information.csv"
TRACKING_DIR = RAW_DATA_DIR / "tracking_parquets"

PROJECT_PATHS = {
    "Project root": PROJECT_ROOT,
    "Raw data directory": RAW_DATA_DIR,
    "PFF data": PFF_PATH,
    "Team information": TEAM_INFO_PATH,
    "Tracking directory": TRACKING_DIR,
}

for path_name, path_value in PROJECT_PATHS.items():
    print(f"{path_name}: {path_value}")
    print(f"  Exists: {path_value.exists()}")

Project root: C:\Users\yvett\OneDrive\Documentos\Francisco Fonseca\nfl-pressure-prediction
  Exists: True
Raw data directory: C:\Users\yvett\OneDrive\Documentos\Francisco Fonseca\nfl-pressure-prediction\data\raw
  Exists: True
PFF data: C:\Users\yvett\OneDrive\Documentos\Francisco Fonseca\nfl-pressure-prediction\data\raw\pffScoutingData.csv
  Exists: True
Team information: C:\Users\yvett\OneDrive\Documentos\Francisco Fonseca\nfl-pressure-prediction\data\raw\team_information.csv
  Exists: True
Tracking directory: C:\Users\yvett\OneDrive\Documentos\Francisco Fonseca\nfl-pressure-prediction\data\raw\tracking_parquets
  Exists: True


In [23]:
import pyarrow.parquet as pq

WEEK_1_PATH = TRACKING_DIR / "week1.parquet"

pff_sample = pd.read_csv(PFF_PATH, nrows=5)
team_info_sample = pd.read_csv(TEAM_INFO_PATH, nrows=5)

week1_parquet = pq.ParquetFile(WEEK_1_PATH)
first_tracking_batch = next(
    week1_parquet.iter_batches(batch_size=5)
)
tracking_sample = first_tracking_batch.to_pandas()

In [24]:
import pyarrow.parquet as pq

WEEK_1_PATH = TRACKING_DIR / "week1.parquet"

pff_sample = pd.read_csv(PFF_PATH, nrows=5)
team_info_sample = pd.read_csv(TEAM_INFO_PATH, nrows=5)

week1_parquet = pq.ParquetFile(WEEK_1_PATH)
first_tracking_batch = next(
    week1_parquet.iter_batches(batch_size=5)
)
tracking_sample = first_tracking_batch.to_pandas()

In [25]:
print(f"PFF sample shape: {pff_sample.shape}")
print(f"Team information sample shape: {team_info_sample.shape}")
print(f"Tracking sample shape: {tracking_sample.shape}")

assert len(pff_sample) == 5
assert len(team_info_sample) == 5
assert len(tracking_sample) == 5

print("Minimal data-read test: PASSED")

PFF sample shape: (5, 15)
Team information sample shape: (5, 4)
Tracking sample shape: (5, 16)
Minimal data-read test: PASSED


## 2. Data inventory and structural validation

### 2.1 Raw-file inventory

This section verifies the files physically available in the raw data directory
before inspecting their schemas or contents.

In [49]:
BYTES_PER_MIB = 1024**2

raw_file_paths = sorted(
    path
    for path in RAW_DATA_DIR.rglob("*")
    if path.is_file()
)

inventory_records = [
    {
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "file_name": path.name,
        "extension": path.suffix.lower() or "[no extension]",
        "size_bytes": path.stat().st_size,
        "size_mib": path.stat().st_size / BYTES_PER_MIB,
    }
    for path in raw_file_paths
]

file_inventory = pd.DataFrame(inventory_records)

file_inventory["size_mib"] = file_inventory["size_mib"].round(3)

print(f"Files found in raw data: {len(file_inventory)}")

file_inventory

Files found in raw data: 11


,relative_path,file_name,extension,size_bytes,size_mib
0,data/raw/.gitkeep,.gitkeep,[no extension],0,0.000
1,data/raw/pffScoutingData.csv,pffScoutingData.csv,.csv,13014038,12.411
2,data/raw/team_information.csv,team_information.csv,.csv,1027,0.001
3,data/raw/tracking_parquets/week1.parquet,week1.parquet,.parquet,32858906,31.337
4,data/raw/tracking_parquets/week2.parquet,week2.parquet,.parquet,30664454,29.244
5,data/raw/tracking_parquets/week3.parquet,week3.parquet,.parquet,16417668,15.657
6,data/raw/tracking_parquets/week4.parquet,week4.parquet,.parquet,16551246,15.784
7,data/raw/tracking_parquets/week5.parquet,week5.parquet,.parquet,31630583,30.165
8,data/raw/tracking_parquets/week6.parquet,week6.parquet,.parquet,32256953,30.763
9,data/raw/tracking_parquets/week7.parquet,week7.parquet,.parquet,28635676,27.309


In [50]:
FIRST_WEEK = 1
LAST_WEEK = 8

EXPECTED_STATIC_FILES = {
    "data/raw/.gitkeep",
    "data/raw/pffScoutingData.csv",
    "data/raw/team_information.csv",
}

EXPECTED_TRACKING_FILES = {
    f"data/raw/tracking_parquets/week{week}.parquet"
    for week in range(FIRST_WEEK, LAST_WEEK + 1)
}

EXPECTED_RAW_FILES = EXPECTED_STATIC_FILES | EXPECTED_TRACKING_FILES
discovered_raw_files = set(file_inventory["relative_path"])

missing_files = sorted(EXPECTED_RAW_FILES - discovered_raw_files)
unexpected_files = sorted(discovered_raw_files - EXPECTED_RAW_FILES)

available_tracking_files = sorted(
    path.name
    for path in TRACKING_DIR.glob("*.parquet")
)

print(f"Expected files: {len(EXPECTED_RAW_FILES)}")
print(f"Discovered files: {len(discovered_raw_files)}")
print(f"Tracking files: {available_tracking_files}")
print(f"Missing files: {missing_files or 'None'}")
print(f"Unexpected files: {unexpected_files or 'None'}")

assert not missing_files, f"Missing files detected: {missing_files}"
assert not unexpected_files, (
    f"Unexpected files detected: {unexpected_files}"
)

print("Raw-file name validation: PASSED")

Expected files: 11
Discovered files: 11
Tracking files: ['week1.parquet', 'week2.parquet', 'week3.parquet', 'week4.parquet', 'week5.parquet', 'week6.parquet', 'week7.parquet', 'week8.parquet']
Missing files: None
Unexpected files: None
Raw-file name validation: PASSED


### 2.2 File-size validation

This section verifies that every expected data file contains data and records
its physical size for reproducibility and quality control.

In [51]:
BYTES_PER_GIB = 1024**3

EXPECTED_DATA_FILES = EXPECTED_RAW_FILES - {
    "data/raw/.gitkeep"
}

data_file_inventory = file_inventory.loc[
    file_inventory["relative_path"].isin(EXPECTED_DATA_FILES)
].copy()

empty_data_files = data_file_inventory.loc[
    data_file_inventory["size_bytes"] == 0,
    "relative_path",
].tolist()

total_raw_size_gib = (
    data_file_inventory["size_bytes"].sum() / BYTES_PER_GIB
)

print(f"Data files checked: {len(data_file_inventory)}")
print(f"Empty data files: {empty_data_files or 'None'}")
print(f"Total raw-data size: {total_raw_size_gib:.3f} GiB")

assert len(data_file_inventory) == len(EXPECTED_DATA_FILES), (
    "The number of data files does not match the expected inventory."
)
assert not empty_data_files, (
    f"Empty data files detected: {empty_data_files}"
)

print("Raw-file size validation: PASSED")

data_file_inventory[
    ["relative_path", "size_bytes", "size_mib"]
].sort_values("relative_path", ignore_index=True)

Data files checked: 10
Empty data files: None
Total raw-data size: 0.240 GiB
Raw-file size validation: PASSED


,relative_path,size_bytes,size_mib
0,data/raw/pffScoutingData.csv,13014038,12.411
1,data/raw/team_information.csv,1027,0.001
2,data/raw/tracking_parquets/week1.parquet,32858906,31.337
3,data/raw/tracking_parquets/week2.parquet,30664454,29.244
4,data/raw/tracking_parquets/week3.parquet,16417668,15.657
5,data/raw/tracking_parquets/week4.parquet,16551246,15.784
6,data/raw/tracking_parquets/week5.parquet,31630583,30.165
7,data/raw/tracking_parquets/week6.parquet,32256953,30.763
8,data/raw/tracking_parquets/week7.parquet,28635676,27.309
9,data/raw/tracking_parquets/week8.parquet,55345725,52.782


### 2.3 Parquet metadata inspection

Parquet metadata provides row counts, column counts, and row-group information
without loading the complete tracking datasets into memory.

In [52]:
tracking_paths = sorted(
    TRACKING_DIR.glob("week*.parquet"),
    key=lambda path: int(path.stem.removeprefix("week")),
)

tracking_metadata_records = []

for tracking_path in tracking_paths:
    parquet_file = pq.ParquetFile(tracking_path)
    parquet_metadata = parquet_file.metadata

    tracking_metadata_records.append(
        {
            "week": int(
                tracking_path.stem.removeprefix("week")
            ),
            "file_name": tracking_path.name,
            "rows": parquet_metadata.num_rows,
            "columns": parquet_metadata.num_columns,
            "row_groups": parquet_metadata.num_row_groups,
            "size_mib": (
                tracking_path.stat().st_size / BYTES_PER_MIB
            ),
        }
    )

tracking_metadata = pd.DataFrame(
    tracking_metadata_records
).sort_values("week", ignore_index=True)

tracking_metadata["size_mib"] = (
    tracking_metadata["size_mib"].round(3)
)

print(tracking_metadata.to_string(index=False))

print(
    "\nTotal tracking rows: "
    f"{tracking_metadata['rows'].sum():,}"
)

 week     file_name    rows  columns  row_groups  size_mib
    1 week1.parquet 1118122       16           1    31.337
    2 week2.parquet 1042774       16           1    29.244
    3 week3.parquet  560908       16           1    15.657
    4 week4.parquet  560917       16           1    15.784
    5 week5.parquet 1074606       16           1    30.165
    6 week6.parquet 1097813       16           1    30.763
    7 week7.parquet  973797       16           1    27.309
    8 week8.parquet 1885241       16           1    52.782

Total tracking rows: 8,314,178


In [53]:
EXPECTED_WEEK_NUMBERS = set(
    range(FIRST_WEEK, LAST_WEEK + 1)
)
EXPECTED_TRACKING_COLUMN_COUNT = 16

observed_week_numbers = set(tracking_metadata["week"])

assert observed_week_numbers == EXPECTED_WEEK_NUMBERS, (
    "The available tracking weeks do not match weeks 1 through 8."
)
assert tracking_metadata["file_name"].is_unique, (
    "Duplicate tracking file names were detected."
)
assert (tracking_metadata["rows"] > 0).all(), (
    "At least one tracking file contains no rows."
)
assert (
    tracking_metadata["columns"]
    == EXPECTED_TRACKING_COLUMN_COUNT
).all(), (
    "Tracking files do not all contain 16 columns."
)
assert (tracking_metadata["row_groups"] > 0).all(), (
    "At least one tracking file contains no row groups."
)
assert (tracking_metadata["size_mib"] > 0).all(), (
    "At least one tracking file has a non-positive size."
)

print("Parquet metadata validation: PASSED")

Parquet metadata validation: PASSED


### 2.4 Tracking schema consistency

The schemas of all weekly tracking files are compared before combining them.
This prevents silent type coercion and structural inconsistencies.

In [54]:
tracking_schemas = {}

for tracking_path in tracking_paths:
    week_number = int(
        tracking_path.stem.removeprefix("week")
    )
    tracking_schemas[week_number] = (
        pq.ParquetFile(tracking_path).schema_arrow
    )

REFERENCE_WEEK = FIRST_WEEK
reference_schema = tracking_schemas[REFERENCE_WEEK]
reference_types = [
    field.type
    for field in reference_schema
]

reference_schema_table = pd.DataFrame(
    {
        "column_position": range(len(reference_schema)),
        "column_name": reference_schema.names,
        "data_type": [
            str(field.type)
            for field in reference_schema
        ],
        "nullable": [
            field.nullable
            for field in reference_schema
        ],
    }
)

schema_comparison_records = []

for week_number, current_schema in tracking_schemas.items():
    current_types = [
        field.type
        for field in current_schema
    ]

    schema_comparison_records.append(
        {
            "week": week_number,
            "column_names_match": (
                current_schema.names
                == reference_schema.names
            ),
            "data_types_match": (
                current_types == reference_types
            ),
            "full_schema_match": current_schema.equals(
                reference_schema,
                check_metadata=False,
            ),
        }
    )

schema_comparison = pd.DataFrame(
    schema_comparison_records
).sort_values("week", ignore_index=True)

print("Reference tracking schema:")
print(reference_schema_table.to_string(index=False))

print("\nSchema comparison by week:")
print(schema_comparison.to_string(index=False))

assert schema_comparison[
    "column_names_match"
].all(), "Tracking column names differ between weeks."

assert schema_comparison[
    "data_types_match"
].all(), "Tracking data types differ between weeks."

assert schema_comparison[
    "full_schema_match"
].all(), "Tracking schemas differ between weeks."

print("\nTracking schema validation: PASSED")

Reference tracking schema:
 column_position   column_name data_type  nullable
               0        gameId     int64      True
               1        playId     int64      True
               2         nflId     int64      True
               3       frameId     int64      True
               4          time    string      True
               5  jerseyNumber     int64      True
               6          team    string      True
               7 playDirection    string      True
               8             x    double      True
               9             y    double      True
              10             s    double      True
              11             a    double      True
              12           dis    double      True
              13             o    double      True
              14           dir    double      True
              15         event    string      True

Schema comparison by week:
 week  column_names_match  data_types_match  full_schema_match
    1          

### 2.5 CSV structure inspection

The two static CSV tables are loaded to validate their dimensions and inferred
data types. No data types are imposed before examining the raw structure.

In [55]:
pff_data = pd.read_csv(
    PFF_PATH,
    low_memory=False,
)

team_information = pd.read_csv(
    TEAM_INFO_PATH,
    low_memory=False,
)

pff_memory_mib = (
    pff_data.memory_usage(deep=True).sum()
    / BYTES_PER_MIB
)
team_memory_mib = (
    team_information.memory_usage(deep=True).sum()
    / BYTES_PER_MIB
)

print(
    "PFF scouting shape: "
    f"{pff_data.shape}"
)
print(
    "PFF memory usage: "
    f"{pff_memory_mib:.3f} MiB"
)

print(
    "\nTeam information shape: "
    f"{team_information.shape}"
)
print(
    "Team information memory usage: "
    f"{team_memory_mib:.3f} MiB"
)

pff_schema_table = pd.DataFrame(
    {
        "column_name": pff_data.columns,
        "data_type": (
            pff_data.dtypes.astype(str).to_numpy()
        ),
    }
)

team_schema_table = pd.DataFrame(
    {
        "column_name": team_information.columns,
        "data_type": (
            team_information.dtypes.astype(str).to_numpy()
        ),
    }
)

print("\nPFF scouting schema:")
print(pff_schema_table.to_string(index=False))

print("\nTeam information schema:")
print(team_schema_table.to_string(index=False))

PFF scouting shape: (188254, 15)
PFF memory usage: 23.793 MiB

Team information shape: (36, 4)
Team information memory usage: 0.002 MiB

PFF scouting schema:
           column_name data_type
                gameId     int64
                playId     int64
                 nflId     int64
              pff_role       str
   pff_positionLinedUp       str
               pff_hit   float64
             pff_hurry   float64
              pff_sack   float64
  pff_beatenByDefender   float64
        pff_hitAllowed   float64
      pff_hurryAllowed   float64
       pff_sackAllowed   float64
pff_nflIdBlockedPlayer   float64
         pff_blockType       str
    pff_backFieldBlock   float64

Team information schema:
column_name data_type
  team_abbr       str
  team_nick       str
 team_color       str
team_color2       str


In [56]:
EXPECTED_PFF_COLUMNS = [
    "gameId",
    "playId",
    "nflId",
    "pff_role",
    "pff_positionLinedUp",
    "pff_hit",
    "pff_hurry",
    "pff_sack",
    "pff_beatenByDefender",
    "pff_hitAllowed",
    "pff_hurryAllowed",
    "pff_sackAllowed",
    "pff_nflIdBlockedPlayer",
    "pff_blockType",
    "pff_backFieldBlock",
]

EXPECTED_TEAM_COLUMNS = [
    "team_abbr",
    "team_nick",
    "team_color",
    "team_color2",
]

assert pff_data.columns.tolist() == EXPECTED_PFF_COLUMNS, (
    "The PFF columns do not match the expected schema."
)
assert team_information.columns.tolist() == EXPECTED_TEAM_COLUMNS, (
    "The team-information columns do not match the expected schema."
)
assert pff_data.columns.is_unique, (
    "Duplicated PFF column names were detected."
)
assert team_information.columns.is_unique, (
    "Duplicated team-information column names were detected."
)

print("CSV column validation: PASSED")

duplicate_team_abbreviations = sorted(
    team_information.loc[
        team_information["team_abbr"].duplicated(keep=False),
        "team_abbr",
    ]
    .astype(str)
    .unique()
    .tolist()
)

team_missing_counts = team_information.isna().sum()

print(
    "\nUnique team abbreviations: "
    f"{team_information['team_abbr'].nunique(dropna=False)}"
)
print(
    "Duplicate team abbreviations: "
    f"{duplicate_team_abbreviations or 'None'}"
)

print("\nMissing values by team column:")
print(team_missing_counts.to_string())

print("\nTeam-information records:")
print(
    team_information.sort_values(
        "team_abbr",
        ignore_index=True,
    ).to_string(index=False)
)

CSV column validation: PASSED

Unique team abbreviations: 36
Duplicate team abbreviations: None

Missing values by team column:
team_abbr      0
team_nick      0
team_color     0
team_color2    0

Team-information records:
team_abbr  team_nick team_color team_color2
      ARI  Cardinals    #97233F     #000000
      ATL    Falcons    #A71930     #000000
      BAL     Ravens    #241773     #9E7C0C
      BUF      Bills    #00338D     #C60C30
      CAR   Panthers    #0085CA     #000000
      CHI      Bears    #0B162A     #C83803
      CIN    Bengals    #FB4F14     #000000
      CLE     Browns    #FF3C00     #311D00
      DAL    Cowboys    #002244     #B0B7BC
      DEN    Broncos    #002244     #FB4F14
      DET      Lions    #0076B6     #B0B7BC
       GB    Packers    #203731     #FFB612
      HOU     Texans    #03202F     #A71930
      IND      Colts    #002C5F     #a5acaf
      JAX    Jaguars    #006778     #000000
       KC     Chiefs    #E31837     #FFB612
       LA       Rams    #0035

### 2.6 PFF key integrity

Each PFF scouting record should represent one unique player within one play.
The composite key is validated for missing values and duplicates.

In [57]:
PFF_PLAY_KEY = [
    "gameId",
    "playId",
]

PFF_PLAYER_PLAY_KEY = [
    "gameId",
    "playId",
    "nflId",
]

pff_key_missing_counts = (
    pff_data[PFF_PLAYER_PLAY_KEY]
    .isna()
    .sum()
)

pff_duplicate_key_mask = pff_data.duplicated(
    subset=PFF_PLAYER_PLAY_KEY,
    keep=False,
)

pff_rows_per_play = (
    pff_data.groupby(
        PFF_PLAY_KEY,
        sort=False,
    )
    .size()
)

unique_games = pff_data["gameId"].nunique()
unique_plays = (
    pff_data[PFF_PLAY_KEY]
    .drop_duplicates()
    .shape[0]
)
unique_players = pff_data["nflId"].nunique()

rows_per_play_values = sorted(
    pff_rows_per_play.unique().tolist()
)

print("Missing values in PFF key:")
print(pff_key_missing_counts.to_string())

print(
    "\nRows involved in duplicated keys: "
    f"{pff_duplicate_key_mask.sum():,}"
)
print(f"Unique games: {unique_games:,}")
print(f"Unique plays: {unique_plays:,}")
print(f"Unique players: {unique_players:,}")
print(
    "Observed rows per play: "
    f"{rows_per_play_values}"
)

assert (pff_key_missing_counts == 0).all(), (
    "Missing values were detected in the PFF key."
)
assert not pff_duplicate_key_mask.any(), (
    "Duplicated player-play keys were detected."
)
assert rows_per_play_values == [22], (
    "Not every PFF play contains exactly 22 players."
)

print("\nPFF key validation: PASSED")

Missing values in PFF key:
gameId    0
playId    0
nflId     0

Rows involved in duplicated keys: 0
Unique games: 122
Unique plays: 8,557
Unique players: 1,679
Observed rows per play: [22]

PFF key validation: PASSED


### 2.7 PFF role distribution

PFF roles are inspected before analyzing missing values because several
scouting variables only apply to specific player responsibilities.

In [58]:
PFF_ROLE_COLUMN = "pff_role"
PASS_RUSH_ROLE = "Pass Rush"

pff_role_summary = (
    pff_data[PFF_ROLE_COLUMN]
    .value_counts(dropna=False)
    .rename_axis(PFF_ROLE_COLUMN)
    .reset_index(name="rows")
)

pff_role_summary["percentage"] = (
    100
    * pff_role_summary["rows"]
    / len(pff_data)
).round(2)

missing_pff_roles = pff_data[
    PFF_ROLE_COLUMN
].isna().sum()

observed_pff_roles = set(
    pff_data[PFF_ROLE_COLUMN].dropna().unique()
)

print(f"Missing PFF roles: {missing_pff_roles:,}")

print("\nPFF role distribution:")
print(pff_role_summary.to_string(index=False))

assert missing_pff_roles == 0, (
    "Missing values were detected in pff_role."
)
assert pff_role_summary["rows"].sum() == len(pff_data), (
    "The role counts do not cover all PFF records."
)
assert PASS_RUSH_ROLE in observed_pff_roles, (
    "The Pass Rush role was not found."
)

print("\nPFF role validation: PASSED")

Missing PFF roles: 0

PFF role distribution:
  pff_role  rows  percentage
  Coverage 57765       30.68
Pass Block 46057       24.47
Pass Route 39513       20.99
 Pass Rush 36362       19.32
      Pass  8557        4.55

PFF role validation: PASSED


### 2.8 Overall PFF missingness

Missing values are quantified before any transformation. At this point,
missingness is descriptive and is not interpreted as zero or absence of an event.

In [59]:
pff_missing_summary = pd.DataFrame(
    {
        "column_name": pff_data.columns,
        "non_missing_count": (
            pff_data.notna().sum().to_numpy()
        ),
        "missing_count": (
            pff_data.isna().sum().to_numpy()
        ),
    }
)

pff_missing_summary["missing_percentage"] = (
    100
    * pff_missing_summary["missing_count"]
    / len(pff_data)
).round(2)

pff_missing_summary = pff_missing_summary.sort_values(
    by=[
        "missing_percentage",
        "column_name",
    ],
    ascending=[
        False,
        True,
    ],
    ignore_index=True,
)

columns_with_missing = (
    pff_missing_summary["missing_count"] > 0
).sum()

print(
    "Columns containing missing values: "
    f"{columns_with_missing} of {pff_data.shape[1]}"
)

print("\nOverall PFF missingness:")
print(pff_missing_summary.to_string(index=False))

Columns containing missing values: 10 of 15

Overall PFF missingness:
           column_name  non_missing_count  missing_count  missing_percentage
pff_nflIdBlockedPlayer              46526         141728               75.29
    pff_backFieldBlock              47903         140351               74.55
         pff_blockType              47904         140350               74.55
  pff_beatenByDefender              48087         140167               74.46
        pff_hitAllowed              48087         140167               74.46
      pff_hurryAllowed              48087         140167               74.46
       pff_sackAllowed              48087         140167               74.46
               pff_hit              94127          94127               50.00
             pff_hurry              94127          94127               50.00
              pff_sack              94127          94127               50.00
                gameId             188254              0                0.00
      

### 2.9 PFF missingness by role

Missingness is analyzed separately for each PFF role to distinguish structural
missing values from potential data-quality problems.

In [60]:
pff_columns_with_missing = (
    pff_missing_summary.loc[
        pff_missing_summary["missing_count"] > 0,
        "column_name",
    ]
    .tolist()
)

pff_role_order = pff_role_summary[
    PFF_ROLE_COLUMN
].tolist()

pff_missing_by_role = (
    pff_data[pff_columns_with_missing]
    .isna()
    .groupby(
        pff_data[PFF_ROLE_COLUMN],
        observed=True,
    )
    .mean()
    .mul(100)
    .round(2)
    .reindex(pff_role_order)
    .transpose()
)

overall_missing_percentage = (
    pff_data[pff_columns_with_missing]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

pff_missing_by_role.insert(
    0,
    "Overall",
    overall_missing_percentage,
)

pff_missing_by_role.index.name = "column_name"
pff_missing_by_role = (
    pff_missing_by_role.reset_index()
)

print("Missing-value percentage by PFF role:")
print(pff_missing_by_role.to_string(index=False))

Missing-value percentage by PFF role:
           column_name  Overall  Coverage  Pass Block  Pass Route  Pass Rush  Pass
pff_nflIdBlockedPlayer    75.29     100.0        3.32       94.94      100.0 100.0
    pff_backFieldBlock    74.55     100.0        0.40       94.86      100.0 100.0
         pff_blockType    74.55     100.0        0.40       94.86      100.0 100.0
  pff_beatenByDefender    74.46     100.0        0.00       94.86      100.0 100.0
        pff_hitAllowed    74.46     100.0        0.00       94.86      100.0 100.0
      pff_hurryAllowed    74.46     100.0        0.00       94.86      100.0 100.0
       pff_sackAllowed    74.46     100.0        0.00       94.86      100.0 100.0
               pff_hit    50.00       0.0      100.00      100.00        0.0 100.0
             pff_hurry    50.00       0.0      100.00      100.00        0.0 100.0
              pff_sack    50.00       0.0      100.00      100.00        0.0 100.0


### 2.10 PFF binary-value validation

All non-missing PFF indicator values are validated against the expected binary
domain before any target construction or missing-value treatment.

In [61]:
PFF_BINARY_COLUMNS = [
    "pff_hit",
    "pff_hurry",
    "pff_sack",
    "pff_beatenByDefender",
    "pff_hitAllowed",
    "pff_hurryAllowed",
    "pff_sackAllowed",
    "pff_backFieldBlock",
]

ALLOWED_BINARY_VALUES = {
    0.0,
    1.0,
}

binary_value_records = []
unexpected_binary_values = {}

for column_name in PFF_BINARY_COLUMNS:
    column_values = pff_data[column_name]

    observed_values = sorted(
        column_values.dropna().unique().tolist()
    )
    unexpected_values = sorted(
        set(observed_values) - ALLOWED_BINARY_VALUES
    )

    binary_value_records.append(
        {
            "column_name": column_name,
            "non_missing": column_values.notna().sum(),
            "missing": column_values.isna().sum(),
            "zero_count": column_values.eq(0).sum(),
            "one_count": column_values.eq(1).sum(),
            "observed_values": observed_values,
            "unexpected_values": (
                unexpected_values or None
            ),
        }
    )

    if unexpected_values:
        unexpected_binary_values[column_name] = (
            unexpected_values
        )

pff_binary_summary = pd.DataFrame(
    binary_value_records
)

print("PFF binary-value summary:")
print(pff_binary_summary.to_string(index=False))

assert not unexpected_binary_values, (
    "Unexpected PFF binary values detected: "
    f"{unexpected_binary_values}"
)

print("\nPFF binary-value validation: PASSED")

PFF binary-value summary:
         column_name  non_missing  missing  zero_count  one_count observed_values unexpected_values
             pff_hit        94127    94127       93284        843      [0.0, 1.0]              None
           pff_hurry        94127    94127       91250       2877      [0.0, 1.0]              None
            pff_sack        94127    94127       93525        602      [0.0, 1.0]              None
pff_beatenByDefender        48087   140167       46143       1944      [0.0, 1.0]              None
      pff_hitAllowed        48087   140167       47543        544      [0.0, 1.0]              None
    pff_hurryAllowed        48087   140167       46004       2083      [0.0, 1.0]              None
     pff_sackAllowed        48087   140167       47717        370      [0.0, 1.0]              None
  pff_backFieldBlock        47903   140351       45894       2009      [0.0, 1.0]              None

PFF binary-value validation: PASSED


### 2.11 PFF categorical-value validation

Categorical columns are checked for empty strings, surrounding whitespace,
and their complete set of observed non-missing values.

In [62]:
PFF_CATEGORICAL_COLUMNS = [
    "pff_role",
    "pff_positionLinedUp",
    "pff_blockType",
]

categorical_quality_records = []
pff_categorical_values = {}

for column_name in PFF_CATEGORICAL_COLUMNS:
    non_missing_values = (
        pff_data[column_name]
        .dropna()
        .astype(str)
    )
    stripped_values = non_missing_values.str.strip()

    blank_count = stripped_values.eq("").sum()
    surrounding_whitespace_count = (
        non_missing_values.ne(stripped_values).sum()
    )

    unique_values = sorted(
        stripped_values.unique().tolist()
    )

    pff_categorical_values[column_name] = unique_values

    categorical_quality_records.append(
        {
            "column_name": column_name,
            "non_missing": len(non_missing_values),
            "missing": pff_data[column_name].isna().sum(),
            "unique_values": len(unique_values),
            "blank_strings": blank_count,
            "surrounding_whitespace": (
                surrounding_whitespace_count
            ),
        }
    )

pff_categorical_quality = pd.DataFrame(
    categorical_quality_records
)

print("PFF categorical-quality summary:")
print(
    pff_categorical_quality.to_string(index=False)
)

for column_name, unique_values in (
    pff_categorical_values.items()
):
    print(f"\n{column_name} values:")
    print(unique_values)

assert (
    pff_categorical_quality["blank_strings"] == 0
).all(), "Blank categorical values were detected."

assert (
    pff_categorical_quality["surrounding_whitespace"] == 0
).all(), "Categorical values with surrounding whitespace were detected."

print("\nPFF categorical-value validation: PASSED")

PFF categorical-quality summary:
        column_name  non_missing  missing  unique_values  blank_strings  surrounding_whitespace
           pff_role       188254        0              5              0                       0
pff_positionLinedUp       188254        0             56              0                       0
      pff_blockType        47904   140350             12              0                       0

pff_role values:
['Coverage', 'Pass', 'Pass Block', 'Pass Route', 'Pass Rush']

pff_positionLinedUp values:
['C', 'DLT', 'DRT', 'FB', 'FB-L', 'FB-R', 'FS', 'FSL', 'FSR', 'HB', 'HB-L', 'HB-R', 'LCB', 'LE', 'LEO', 'LG', 'LILB', 'LLB', 'LOLB', 'LT', 'LWR', 'MLB', 'NLT', 'NRT', 'NT', 'QB', 'RCB', 'RE', 'REO', 'RG', 'RILB', 'RLB', 'ROLB', 'RT', 'RWR', 'SCBL', 'SCBR', 'SCBiL', 'SCBiR', 'SCBoL', 'SCBoR', 'SLWR', 'SLiWR', 'SLoWR', 'SRWR', 'SRiWR', 'SRoWR', 'SS', 'SSL', 'SSR', 'TE-L', 'TE-R', 'TE-iL', 'TE-iR', 'TE-oL', 'TE-oR']

pff_blockType values:
['BH', 'CH', 'CL', 'NB', 'PA', 'PP

### 2.12 Tracking missingness from Parquet metadata

Column-level Parquet statistics are used to quantify missing tracking values
without loading the complete weekly datasets into memory.

In [63]:
tracking_null_metadata_records = []

for tracking_path in tracking_paths:
    week_number = int(
        tracking_path.stem.removeprefix("week")
    )
    parquet_file = pq.ParquetFile(tracking_path)
    file_metadata = parquet_file.metadata

    for row_group_index in range(
        file_metadata.num_row_groups
    ):
        row_group_metadata = file_metadata.row_group(
            row_group_index
        )

        for column_index, column_name in enumerate(
            reference_schema.names
        ):
            column_metadata = row_group_metadata.column(
                column_index
            )
            column_statistics = (
                column_metadata.statistics
            )

            null_count = (
                column_statistics.null_count
                if column_statistics is not None
                else None
            )

            tracking_null_metadata_records.append(
                {
                    "week": week_number,
                    "row_group": row_group_index,
                    "column_name": column_name,
                    "row_count": (
                        row_group_metadata.num_rows
                    ),
                    "null_count": null_count,
                    "statistics_available": (
                        null_count is not None
                    ),
                }
            )

tracking_null_metadata = pd.DataFrame(
    tracking_null_metadata_records
)

all_null_statistics_available = (
    tracking_null_metadata[
        "statistics_available"
    ].all()
)

print(
    "Null-count statistics available for all columns: "
    f"{all_null_statistics_available}"
)

if all_null_statistics_available:
    tracking_missing_summary = (
        tracking_null_metadata.groupby(
            "column_name",
            sort=False,
            as_index=False,
        )
        .agg(
            total_rows=("row_count", "sum"),
            missing_count=("null_count", "sum"),
        )
    )

    tracking_missing_summary["non_missing_count"] = (
        tracking_missing_summary["total_rows"]
        - tracking_missing_summary["missing_count"]
    )

    tracking_missing_summary["missing_percentage"] = (
        100
        * tracking_missing_summary["missing_count"]
        / tracking_missing_summary["total_rows"]
    ).round(2)

    tracking_missing_summary = (
        tracking_missing_summary.sort_values(
            by=[
                "missing_percentage",
                "column_name",
            ],
            ascending=[
                False,
                True,
            ],
            ignore_index=True,
        )
    )

    print("\nOverall tracking missingness:")
    print(
        tracking_missing_summary.to_string(
            index=False
        )
    )
else:
    unavailable_statistics = (
        tracking_null_metadata.loc[
            ~tracking_null_metadata[
                "statistics_available"
            ],
            [
                "week",
                "row_group",
                "column_name",
            ],
        ]
    )

    print("\nUnavailable null-count statistics:")
    print(
        unavailable_statistics.to_string(
            index=False
        )
    )

Null-count statistics available for all columns: True

Overall tracking missingness:
  column_name  total_rows  missing_count  non_missing_count  missing_percentage
          dir     8314178         361486            7952692                4.35
 jerseyNumber     8314178         361486            7952692                4.35
        nflId     8314178         361486            7952692                4.35
            o     8314178         361486            7952692                4.35
            a     8314178              0            8314178                0.00
          dis     8314178              0            8314178                0.00
        event     8314178              0            8314178                0.00
      frameId     8314178              0            8314178                0.00
       gameId     8314178              0            8314178                0.00
playDirection     8314178              0            8314178                0.00
       playId     8314178          

### 2.13 Tracking entity-missingness validation

Tracking data is scanned in memory-efficient batches to verify whether missing
identity and orientation fields belong exclusively to football records.

In [64]:
TRACKING_BATCH_SIZE = 250_000

TRACKING_IDENTITY_COLUMNS = [
    "nflId",
    "jerseyNumber",
    "team",
    "o",
    "dir",
]

TRACKING_ENTITY_MISSING_COLUMNS = [
    "nflId",
    "jerseyNumber",
    "o",
    "dir",
]

EXPECTED_TRACKING_TEAM_VALUES = {
    "home",
    "away",
    "football",
}

tracking_identity_records = []
tracking_team_value_counts = {}

for tracking_path in tracking_paths:
    week_number = int(
        tracking_path.stem.removeprefix("week")
    )
    parquet_file = pq.ParquetFile(tracking_path)

    rows_scanned = 0
    weekly_team_counts = {}
    partial_missing_rows = 0
    nonfootball_with_missing = 0
    football_without_all_missing = 0

    for record_batch in parquet_file.iter_batches(
        batch_size=TRACKING_BATCH_SIZE,
        columns=TRACKING_IDENTITY_COLUMNS,
    ):
        batch_data = record_batch.to_pandas()
        rows_scanned += len(batch_data)

        football_mask = batch_data["team"].eq(
            "football"
        )

        missing_matrix = batch_data[
            TRACKING_ENTITY_MISSING_COLUMNS
        ].isna()

        any_missing_mask = missing_matrix.any(axis=1)
        all_missing_mask = missing_matrix.all(axis=1)

        partial_missing_rows += int(
            (
                any_missing_mask
                & ~all_missing_mask
            ).sum()
        )

        nonfootball_with_missing += int(
            (
                ~football_mask
                & any_missing_mask
            ).sum()
        )

        football_without_all_missing += int(
            (
                football_mask
                & ~all_missing_mask
            ).sum()
        )

        batch_team_counts = (
            batch_data["team"]
            .value_counts(dropna=False)
        )

        for team_value, count in (
            batch_team_counts.items()
        ):
            team_key = (
                "<MISSING>"
                if pd.isna(team_value)
                else str(team_value)
            )

            weekly_team_counts[team_key] = (
                weekly_team_counts.get(team_key, 0)
                + int(count)
            )

            tracking_team_value_counts[team_key] = (
                tracking_team_value_counts.get(
                    team_key,
                    0,
                )
                + int(count)
            )

    football_rows = weekly_team_counts.get(
        "football",
        0,
    )
    player_rows = rows_scanned - football_rows

    unexpected_team_rows = sum(
        count
        for team_value, count
        in weekly_team_counts.items()
        if team_value
        not in EXPECTED_TRACKING_TEAM_VALUES
    )

    tracking_identity_records.append(
        {
            "week": week_number,
            "rows_scanned": rows_scanned,
            "home_rows": weekly_team_counts.get(
                "home",
                0,
            ),
            "away_rows": weekly_team_counts.get(
                "away",
                0,
            ),
            "football_rows": football_rows,
            "player_rows_per_football_row": round(
                player_rows / football_rows,
                4,
            ),
            "unexpected_team_rows": (
                unexpected_team_rows
            ),
            "partial_missing_rows": (
                partial_missing_rows
            ),
            "nonfootball_with_missing": (
                nonfootball_with_missing
            ),
            "football_without_all_missing": (
                football_without_all_missing
            ),
        }
    )

tracking_identity_summary = pd.DataFrame(
    tracking_identity_records
).sort_values("week", ignore_index=True)

tracking_team_summary = pd.DataFrame(
    [
        {
            "team_value": team_value,
            "rows": count,
        }
        for team_value, count
        in tracking_team_value_counts.items()
    ]
).sort_values("team_value", ignore_index=True)

print("Tracking team values:")
print(tracking_team_summary.to_string(index=False))

print("\nTracking identity validation by week:")
print(
    tracking_identity_summary.to_string(
        index=False
    )
)

assert (
    tracking_identity_summary["rows_scanned"].sum()
    == tracking_metadata["rows"].sum()
), "The batch scan did not cover every tracking row."

assert (
    tracking_identity_summary[
        "partial_missing_rows"
    ] == 0
).all(), "Partially missing identity records were detected."

assert (
    tracking_identity_summary[
        "nonfootball_with_missing"
    ] == 0
).all(), "Player records with missing identity fields were detected."

assert (
    tracking_identity_summary[
        "football_without_all_missing"
    ] == 0
).all(), "Football records with player identity fields were detected."

print("\nTracking entity-missingness validation: PASSED")

Tracking team values:
team_value   rows
       ARI 237842
       ATL 237017
       BAL 254166
       BUF 241428
       CAR 256014
       CHI 228382
       CIN 270413
       CLE 238260
       DAL 241329
       DEN 257323
       DET 258885
        GB 236973
       HOU 224719
       IND 249546
       JAX 228668
        KC 286132
        LA 259391
       LAC 232837
        LV 245993
       MIA 277596
       MIN 233596
        NE 257686
        NO 228833
       NYG 271645
       NYJ 237831
       PHI 244255
       PIT 207988
       SEA 253671
        SF 208549
        TB 282194
       TEN 274813
       WAS 288717
  football 361486

Tracking identity validation by week:
 week  rows_scanned  home_rows  away_rows  football_rows  player_rows_per_football_row  unexpected_team_rows  partial_missing_rows  nonfootball_with_missing  football_without_all_missing
    1       1118122          0          0          48614                       22.0000               1069508                     0          

#### Correction: tracking team encoding

Unlike some NFL tracking datasets, this dataset stores team abbreviations
instead of the generic values `home` and `away`.

In [65]:
FOOTBALL_TEAM_VALUE = "football"

TEAM_METADATA_ABBREVIATIONS = set(
    team_information["team_abbr"]
)

EXPECTED_TRACKING_TEAM_VALUES = (
    TEAM_METADATA_ABBREVIATIONS
    | {FOOTBALL_TEAM_VALUE}
)

observed_tracking_team_values = set(
    tracking_team_value_counts
)

observed_player_team_values = (
    observed_tracking_team_values
    - {FOOTBALL_TEAM_VALUE}
)

unexpected_tracking_team_values = sorted(
    observed_tracking_team_values
    - EXPECTED_TRACKING_TEAM_VALUES
)

unused_team_metadata_values = sorted(
    TEAM_METADATA_ABBREVIATIONS
    - observed_player_team_values
)

tracking_identity_summary_corrected = (
    tracking_identity_summary.drop(
        columns=[
            "home_rows",
            "away_rows",
            "unexpected_team_rows",
        ],
        errors="ignore",
    )
)

print(
    "Observed player-team abbreviations: "
    f"{len(observed_player_team_values)}"
)
print(
    "Unexpected tracking team values: "
    f"{unexpected_tracking_team_values or 'None'}"
)
print(
    "Unused team-information aliases: "
    f"{unused_team_metadata_values or 'None'}"
)

assert FOOTBALL_TEAM_VALUE in (
    observed_tracking_team_values
), "The football tracking label was not found."

assert not unexpected_tracking_team_values, (
    "Tracking contains team abbreviations absent "
    "from team_information."
)

assert len(observed_player_team_values) == 32, (
    "The tracking data does not contain 32 player teams."
)

print("\nCorrected tracking identity summary:")
print(
    tracking_identity_summary_corrected.to_string(
        index=False
    )
)

print("\nTracking team-value validation: PASSED")

Observed player-team abbreviations: 32
Unexpected tracking team values: None
Unused team-information aliases: ['LAR', 'OAK', 'SD', 'STL']

Corrected tracking identity summary:
 week  rows_scanned  football_rows  player_rows_per_football_row  partial_missing_rows  nonfootball_with_missing  football_without_all_missing
    1       1118122          48614                       22.0000                     0                         0                             0
    2       1042774          45338                       22.0000                     0                         0                             0
    3        560908          24376                       22.0107                     0                         0                             0
    4        560917          24399                       21.9893                     0                         0                             0
    5       1074606          46722                       22.0000                     0                       

### 2.14 Weekly tracking-partition integrity

Games and plays should belong to only one weekly tracking file. Overlap between
weekly partitions could compromise the planned temporal train-validation-test split.

In [66]:
TRACKING_PARTITION_COLUMNS = [
    "gameId",
    "playId",
]

tracking_game_ids_by_week = {}
tracking_play_keys_by_week = {}
tracking_partition_records = []

for tracking_path in tracking_paths:
    week_number = int(
        tracking_path.stem.removeprefix("week")
    )
    parquet_file = pq.ParquetFile(tracking_path)

    weekly_game_ids = set()
    weekly_play_keys = set()

    for record_batch in parquet_file.iter_batches(
        batch_size=TRACKING_BATCH_SIZE,
        columns=TRACKING_PARTITION_COLUMNS,
    ):
        batch_keys = record_batch.to_pandas()

        weekly_game_ids.update(
            batch_keys["gameId"].unique().tolist()
        )

        unique_batch_play_keys = (
            batch_keys[
                TRACKING_PARTITION_COLUMNS
            ]
            .drop_duplicates()
        )

        weekly_play_keys.update(
            unique_batch_play_keys.itertuples(
                index=False,
                name=None,
            )
        )

    tracking_game_ids_by_week[week_number] = (
        weekly_game_ids
    )
    tracking_play_keys_by_week[week_number] = (
        weekly_play_keys
    )

    tracking_partition_records.append(
        {
            "week": week_number,
            "unique_games": len(weekly_game_ids),
            "unique_plays": len(weekly_play_keys),
            "minimum_game_id": min(weekly_game_ids),
            "maximum_game_id": max(weekly_game_ids),
        }
    )

tracking_partition_summary = pd.DataFrame(
    tracking_partition_records
).sort_values("week", ignore_index=True)

partition_overlap_records = []
week_numbers = sorted(
    tracking_game_ids_by_week
)

for first_index, first_week in enumerate(
    week_numbers
):
    for second_week in week_numbers[
        first_index + 1:
    ]:
        overlapping_game_ids = sorted(
            tracking_game_ids_by_week[first_week]
            & tracking_game_ids_by_week[second_week]
        )

        overlapping_play_keys = sorted(
            tracking_play_keys_by_week[first_week]
            & tracking_play_keys_by_week[second_week]
        )

        if (
            overlapping_game_ids
            or overlapping_play_keys
        ):
            partition_overlap_records.append(
                {
                    "first_week": first_week,
                    "second_week": second_week,
                    "overlapping_games": len(
                        overlapping_game_ids
                    ),
                    "overlapping_plays": len(
                        overlapping_play_keys
                    ),
                    "game_id_sample": (
                        overlapping_game_ids[:5]
                    ),
                    "play_key_sample": (
                        overlapping_play_keys[:5]
                    ),
                }
            )

all_tracking_game_ids = set().union(
    *tracking_game_ids_by_week.values()
)
all_tracking_play_keys = set().union(
    *tracking_play_keys_by_week.values()
)

print("Tracking partition summary:")
print(
    tracking_partition_summary.to_string(
        index=False
    )
)

print(
    "\nTotal unique tracking games: "
    f"{len(all_tracking_game_ids):,}"
)
print(
    "Total unique tracking plays: "
    f"{len(all_tracking_play_keys):,}"
)

print("\nCross-week partition overlaps:")

if partition_overlap_records:
    tracking_partition_overlaps = pd.DataFrame(
        partition_overlap_records
    )
    print(
        tracking_partition_overlaps.to_string(
            index=False
        )
    )
else:
    tracking_partition_overlaps = pd.DataFrame()
    print("None")

Tracking partition summary:
 week  unique_games  unique_plays  minimum_game_id  maximum_game_id
    1            16          1175       2021090900       2021091300
    2            16          1067       2021091600       2021092000
    3             8           549       2021092300       2021092606
    4             9           593       2021092606       2021092700
    5            16          1113       2021093000       2021100400
    6            16          1108       2021100700       2021101100
    7            14          1004       2021101400       2021101800
    8            28          1949       2021102100       2021110100

Total unique tracking games: 122
Total unique tracking plays: 8,557

Cross-week partition overlaps:
 first_week  second_week  overlapping_games  overlapping_plays game_id_sample      play_key_sample
          3            4                  1                  1   [2021092606] [(2021092606, 2946)]


### 2.15 Shared-play investigation

The play shared by source files 3 and 4 is inspected to determine whether it
is duplicated, incomplete, or physically divided between both Parquet files.

In [67]:
OVERLAP_GAME_ID = 2021092606
OVERLAP_PLAY_ID = 2946
OVERLAP_SOURCE_FILE_NUMBERS = [
    3,
    4,
]

OVERLAP_COLUMNS = [
    "gameId",
    "playId",
    "frameId",
    "nflId",
    "team",
    "event",
]

tracking_path_by_file_number = {
    int(path.stem.removeprefix("week")): path
    for path in tracking_paths
}

overlap_tracking_parts = []

for source_file_number in (
    OVERLAP_SOURCE_FILE_NUMBERS
):
    tracking_path = (
        tracking_path_by_file_number[
            source_file_number
        ]
    )

    overlap_table = pq.read_table(
        tracking_path,
        columns=OVERLAP_COLUMNS,
        filters=[
            (
                "gameId",
                "=",
                OVERLAP_GAME_ID,
            ),
            (
                "playId",
                "=",
                OVERLAP_PLAY_ID,
            ),
        ],
    )

    overlap_part = overlap_table.to_pandas()
    overlap_part["source_file_number"] = (
        source_file_number
    )

    overlap_tracking_parts.append(
        overlap_part
    )

overlap_tracking = pd.concat(
    overlap_tracking_parts,
    ignore_index=True,
)

overlap_source_summary = (
    overlap_tracking.groupby(
        "source_file_number",
        as_index=False,
    )
    .agg(
        rows=("frameId", "size"),
        unique_frames=("frameId", "nunique"),
        minimum_frame_id=("frameId", "min"),
        maximum_frame_id=("frameId", "max"),
        unique_player_ids=("nflId", "nunique"),
        football_rows=(
            "team",
            lambda values: values.eq(
                FOOTBALL_TEAM_VALUE
            ).sum(),
        ),
    )
)

OVERLAP_FRAME_KEY = [
    "gameId",
    "playId",
    "frameId",
]

OVERLAP_ENTITY_FRAME_KEY = [
    *OVERLAP_FRAME_KEY,
    "nflId",
]

overlap_rows_per_frame = (
    overlap_tracking.groupby(
        OVERLAP_FRAME_KEY,
        sort=True,
    )
    .size()
)

overlap_frame_size_distribution = (
    overlap_rows_per_frame
    .value_counts()
    .sort_index()
    .rename_axis("rows_per_frame")
    .reset_index(name="frame_count")
)

frame_source_counts = (
    overlap_tracking.groupby(
        OVERLAP_FRAME_KEY
    )["source_file_number"]
    .nunique()
)

frames_split_between_files = int(
    frame_source_counts.gt(1).sum()
)

duplicated_entity_frame_rows = int(
    overlap_tracking.duplicated(
        subset=OVERLAP_ENTITY_FRAME_KEY,
        keep=False,
    ).sum()
)

source_contribution_by_frame = (
    overlap_tracking.groupby(
        [
            "frameId",
            "source_file_number",
        ]
    )
    .size()
    .unstack(fill_value=0)
)

source_contribution_patterns = (
    source_contribution_by_frame
    .value_counts()
    .rename("frame_count")
    .reset_index()
)

print("Shared-play contribution by source file:")
print(
    overlap_source_summary.to_string(
        index=False
    )
)

print("\nSource contribution patterns per frame:")
print(
    source_contribution_patterns.to_string(
        index=False
    )
)

print("\nCombined rows-per-frame distribution:")
print(
    overlap_frame_size_distribution.to_string(
        index=False
    )
)

print(
    "\nFrames represented in both files: "
    f"{frames_split_between_files:,}"
)
print(
    "Duplicated entity-frame rows after combining: "
    f"{duplicated_entity_frame_rows:,}"
)

Shared-play contribution by source file:
 source_file_number  rows  unique_frames  minimum_frame_id  maximum_frame_id  unique_player_ids  football_rows
                  3   260             37                 1                37                  8              0
                  4   591             37                 1                37                 15             37

Source contribution patterns per frame:
 3  4  frame_count
 7 16           36
 8 15            1

Combined rows-per-frame distribution:
 rows_per_frame  frame_count
             23           37

Frames represented in both files: 37
Duplicated entity-frame rows after combining: 0


### 2.16 Game-date and source-file catalog

Game dates are extracted from `gameId` to distinguish physical Parquet file
numbers from actual NFL calendar weeks.

In [68]:
tracking_game_file_records = []

for source_file_number, game_ids in (
    tracking_game_ids_by_week.items()
):
    for game_id in game_ids:
        tracking_game_file_records.append(
            {
                "source_file_number": (
                    source_file_number
                ),
                "gameId": game_id,
            }
        )

tracking_game_file_map = pd.DataFrame(
    tracking_game_file_records
)

tracking_game_file_map["game_date"] = (
    pd.to_datetime(
        tracking_game_file_map[
            "gameId"
        ]
        .astype(str)
        .str[:8],
        format="%Y%m%d",
    )
)

tracking_game_catalog = (
    tracking_game_file_map.groupby(
        "gameId",
        as_index=False,
    )
    .agg(
        game_date=("game_date", "first"),
        source_file_count=(
            "source_file_number",
            "nunique",
        ),
        source_file_numbers=(
            "source_file_number",
            lambda values: sorted(set(values)),
        ),
    )
)

tracking_game_date_summary = (
    tracking_game_file_map.groupby(
        "game_date",
        as_index=False,
    )
    .agg(
        unique_games=("gameId", "nunique"),
        source_file_numbers=(
            "source_file_number",
            lambda values: sorted(set(values)),
        ),
    )
    .sort_values(
        "game_date",
        ignore_index=True,
    )
)

games_in_multiple_source_files = (
    tracking_game_catalog.loc[
        tracking_game_catalog[
            "source_file_count"
        ] > 1
    ]
)

print("Unique games by date:")
print(
    tracking_game_date_summary.to_string(
        index=False
    )
)

print("\nGames stored in multiple source files:")
if games_in_multiple_source_files.empty:
    print("None")
else:
    print(
        games_in_multiple_source_files.to_string(
            index=False
        )
    )

Unique games by date:
 game_date  unique_games source_file_numbers
2021-09-09             1                 [1]
2021-09-12            14                 [1]
2021-09-13             1                 [1]
2021-09-16             1                 [2]
2021-09-19            14                 [2]
2021-09-20             1                 [2]
2021-09-23             1                 [3]
2021-09-26            14              [3, 4]
2021-09-27             1                 [4]
2021-09-30             1                 [5]
2021-10-03            14                 [5]
2021-10-04             1                 [5]
2021-10-07             1                 [6]
2021-10-10            14                 [6]
2021-10-11             1                 [6]
2021-10-14             1                 [7]
2021-10-17            12                 [7]
2021-10-18             1                 [7]
2021-10-21             1                 [8]
2021-10-24            11                 [8]
2021-10-25             1         

### 2.17 Actual NFL week mapping

Actual NFL weeks are assigned at the game level using calendar-date ranges.
Physical Parquet file numbers are retained only for data lineage.

In [69]:
ACTUAL_WEEK_DATE_RANGES = {
    1: (
        pd.Timestamp("2021-09-09"),
        pd.Timestamp("2021-09-13"),
    ),
    2: (
        pd.Timestamp("2021-09-16"),
        pd.Timestamp("2021-09-20"),
    ),
    3: (
        pd.Timestamp("2021-09-23"),
        pd.Timestamp("2021-09-27"),
    ),
    4: (
        pd.Timestamp("2021-09-30"),
        pd.Timestamp("2021-10-04"),
    ),
    5: (
        pd.Timestamp("2021-10-07"),
        pd.Timestamp("2021-10-11"),
    ),
    6: (
        pd.Timestamp("2021-10-14"),
        pd.Timestamp("2021-10-18"),
    ),
    7: (
        pd.Timestamp("2021-10-21"),
        pd.Timestamp("2021-10-25"),
    ),
    8: (
        pd.Timestamp("2021-10-28"),
        pd.Timestamp("2021-11-01"),
    ),
}

EXPECTED_GAMES_BY_ACTUAL_WEEK = {
    1: 16,
    2: 16,
    3: 16,
    4: 16,
    5: 16,
    6: 14,
    7: 13,
    8: 15,
}


def assign_actual_week(game_date):
    matching_weeks = [
        week_number
        for week_number, (
            start_date,
            end_date,
        ) in ACTUAL_WEEK_DATE_RANGES.items()
        if start_date <= game_date <= end_date
    ]

    if len(matching_weeks) != 1:
        return pd.NA

    return matching_weeks[0]


tracking_game_catalog["actual_week"] = (
    tracking_game_catalog["game_date"]
    .apply(assign_actual_week)
    .astype("Int64")
)

unassigned_games = tracking_game_catalog.loc[
    tracking_game_catalog["actual_week"].isna(),
    [
        "gameId",
        "game_date",
        "source_file_numbers",
    ],
]

tracking_actual_week_summary = (
    tracking_game_catalog.groupby(
        "actual_week",
        as_index=False,
        dropna=False,
    )
    .agg(
        unique_games=("gameId", "nunique"),
        first_game_date=("game_date", "min"),
        last_game_date=("game_date", "max"),
        source_file_numbers=(
            "source_file_numbers",
            lambda values: sorted(
                {
                    file_number
                    for file_numbers in values
                    for file_number in file_numbers
                }
            ),
        ),
    )
)

observed_games_by_actual_week = (
    tracking_actual_week_summary
    .set_index("actual_week")["unique_games"]
    .to_dict()
)

assert unassigned_games.empty, (
    "At least one game could not be assigned "
    "to exactly one actual NFL week."
)
assert (
    observed_games_by_actual_week
    == EXPECTED_GAMES_BY_ACTUAL_WEEK
), (
    "Observed game counts do not match the "
    "expected schedule by actual week."
)

tracking_game_week_map = (
    tracking_game_catalog[
        [
            "gameId",
            "game_date",
            "actual_week",
        ]
    ]
    .sort_values("gameId", ignore_index=True)
)

print("Actual NFL week summary:")
print(
    tracking_actual_week_summary.to_string(
        index=False
    )
)

print(
    "\nGames in week mapping: "
    f"{len(tracking_game_week_map):,}"
)
print("Actual NFL week mapping: PASSED")

Actual NFL week summary:
 actual_week  unique_games first_game_date last_game_date source_file_numbers
           1            16      2021-09-09     2021-09-13                 [1]
           2            16      2021-09-16     2021-09-20                 [2]
           3            16      2021-09-23     2021-09-27              [3, 4]
           4            16      2021-09-30     2021-10-04                 [5]
           5            16      2021-10-07     2021-10-11                 [6]
           6            14      2021-10-14     2021-10-18                 [7]
           7            13      2021-10-21     2021-10-25                 [8]
           8            15      2021-10-28     2021-11-01                 [8]

Games in week mapping: 122
Actual NFL week mapping: PASSED


### 2.18 Tracking entity-frame key integrity

Tracking row uniqueness is validated at the entity-frame level. Frame-level
counts are accumulated across physical files so split plays are reconstructed.

In [70]:
TRACKING_FRAME_KEY = [
    "gameId",
    "playId",
    "frameId",
]

TRACKING_ENTITY_FRAME_KEY = [
    *TRACKING_FRAME_KEY,
    "nflId",
]

EXPECTED_ENTITIES_PER_FRAME = 23

tracking_key_records = []
global_frame_row_counts = {}

for source_file_number, tracking_path in (
    tracking_path_by_file_number.items()
):
    tracking_key_data = pd.read_parquet(
        tracking_path,
        columns=TRACKING_ENTITY_FRAME_KEY,
        engine="pyarrow",
    )

    duplicated_key_mask = (
        tracking_key_data.duplicated(
            subset=TRACKING_ENTITY_FRAME_KEY,
            keep=False,
        )
    )

    file_frame_counts = (
        tracking_key_data.groupby(
            TRACKING_FRAME_KEY,
            sort=False,
        )
        .size()
    )

    for frame_key, row_count in (
        file_frame_counts.items()
    ):
        global_frame_row_counts[frame_key] = (
            global_frame_row_counts.get(
                frame_key,
                0,
            )
            + int(row_count)
        )

    tracking_key_records.append(
        {
            "source_file_number": (
                source_file_number
            ),
            "rows_checked": len(
                tracking_key_data
            ),
            "frame_keys_in_file": len(
                file_frame_counts
            ),
            "duplicated_entity_frame_rows": int(
                duplicated_key_mask.sum()
            ),
        }
    )

tracking_key_summary = pd.DataFrame(
    tracking_key_records
).sort_values(
    "source_file_number",
    ignore_index=True,
)

tracking_global_frame_counts = pd.DataFrame(
    [
        {
            "gameId": frame_key[0],
            "playId": frame_key[1],
            "frameId": frame_key[2],
            "rows_per_frame": row_count,
        }
        for frame_key, row_count
        in global_frame_row_counts.items()
    ]
)

tracking_frame_size_distribution = (
    tracking_global_frame_counts[
        "rows_per_frame"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("rows_per_frame")
    .reset_index(name="frame_count")
)

print("Tracking key validation by source file:")
print(
    tracking_key_summary.to_string(
        index=False
    )
)

print("\nGlobal rows-per-frame distribution:")
print(
    tracking_frame_size_distribution.to_string(
        index=False
    )
)

print(
    "\nGlobal unique frames: "
    f"{len(tracking_global_frame_counts):,}"
)

assert (
    tracking_key_summary[
        "duplicated_entity_frame_rows"
    ].sum()
    == 0
), "Duplicated entity-frame keys exist within a source file."

assert duplicated_entity_frame_rows == 0, (
    "Duplicated entity-frame keys exist "
    "across source files 3 and 4."
)

assert len(tracking_global_frame_counts) == (
    tracking_team_value_counts[
        FOOTBALL_TEAM_VALUE
    ]
), (
    "The number of unique frames does not "
    "match the number of football records."
)

assert (
    tracking_global_frame_counts[
        "rows_per_frame"
    ] == EXPECTED_ENTITIES_PER_FRAME
).all(), (
    "At least one tracking frame does not "
    "contain exactly 23 entities."
)

print("\nTracking entity-frame validation: PASSED")

Tracking key validation by source file:
 source_file_number  rows_checked  frame_keys_in_file  duplicated_entity_frame_rows
                  1       1118122               48614                             0
                  2       1042774               45338                             0
                  3        560908               24413                             0
                  4        560917               24399                             0
                  5       1074606               46722                             0
                  6       1097813               47731                             0
                  7        973797               42339                             0
                  8       1885241               81967                             0

Global rows-per-frame distribution:
 rows_per_frame  frame_count
             23       361486

Global unique frames: 361,486

Tracking entity-frame validation: PASSED


### 2.19 Tracking frame-sequence validation

Frame identifiers are checked within every play to ensure that sequences start
at frame 1 and contain no internal gaps.

In [71]:
tracking_play_frame_summary = (
    tracking_global_frame_counts.groupby(
        [
            "gameId",
            "playId",
        ],
        as_index=False,
    )
    .agg(
        frame_count=("frameId", "nunique"),
        minimum_frame_id=("frameId", "min"),
        maximum_frame_id=("frameId", "max"),
    )
)

tracking_play_frame_summary[
    "expected_contiguous_frame_count"
] = (
    tracking_play_frame_summary[
        "maximum_frame_id"
    ]
    - tracking_play_frame_summary[
        "minimum_frame_id"
    ]
    + 1
)

tracking_play_frame_summary[
    "has_contiguous_frames"
] = (
    tracking_play_frame_summary["frame_count"]
    == tracking_play_frame_summary[
        "expected_contiguous_frame_count"
    ]
)

plays_not_starting_at_one = (
    tracking_play_frame_summary.loc[
        tracking_play_frame_summary[
            "minimum_frame_id"
        ] != 1
    ]
)

plays_with_frame_gaps = (
    tracking_play_frame_summary.loc[
        ~tracking_play_frame_summary[
            "has_contiguous_frames"
        ]
    ]
)

frame_count_description = (
    tracking_play_frame_summary["frame_count"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

print(
    "Plays checked: "
    f"{len(tracking_play_frame_summary):,}"
)
print(
    "Plays not starting at frame 1: "
    f"{len(plays_not_starting_at_one):,}"
)
print(
    "Plays containing frame gaps: "
    f"{len(plays_with_frame_gaps):,}"
)

print("\nFrames per play:")
print(frame_count_description.to_string())

assert len(tracking_play_frame_summary) == len(
    all_tracking_play_keys
), "The global play count is inconsistent."

assert plays_not_starting_at_one.empty, (
    "At least one play does not start at frame 1."
)

assert plays_with_frame_gaps.empty, (
    "At least one play contains missing frame IDs."
)

print("\nTracking frame-sequence validation: PASSED")

Plays checked: 8,557
Plays not starting at frame 1: 0
Plays containing frame gaps: 0

Frames per play:
count    8557.000000
mean       42.244478
std        12.510596
min        19.000000
25%        34.000000
50%        39.000000
75%        47.000000
95%        66.000000
99%        83.440000
max       203.000000

Tracking frame-sequence validation: PASSED


### 2.20 Frame-level event inspection

One football record exists per frame, so football rows are used to inspect
frame-level events without counting the same event once per tracked entity.

In [72]:
FOOTBALL_FRAME_COLUMNS = [
    "gameId",
    "playId",
    "frameId",
    "team",
    "event",
]

football_frame_parts = []

for source_file_number, tracking_path in (
    tracking_path_by_file_number.items()
):
    football_table = pq.read_table(
        tracking_path,
        columns=FOOTBALL_FRAME_COLUMNS,
        filters=[
            (
                "team",
                "=",
                FOOTBALL_TEAM_VALUE,
            )
        ],
    )

    football_part = football_table.to_pandas()
    football_part["source_file_number"] = (
        source_file_number
    )

    football_frame_parts.append(
        football_part
    )

tracking_football_frames = pd.concat(
    football_frame_parts,
    ignore_index=True,
)

duplicated_football_frames = int(
    tracking_football_frames.duplicated(
        subset=TRACKING_FRAME_KEY,
        keep=False,
    ).sum()
)

non_missing_event_values = (
    tracking_football_frames["event"]
    .dropna()
    .astype(str)
)

stripped_event_values = (
    non_missing_event_values.str.strip()
)

empty_event_count = int(
    stripped_event_values.eq("").sum()
)

event_whitespace_count = int(
    non_missing_event_values.ne(
        stripped_event_values
    ).sum()
)

event_display_values = (
    tracking_football_frames["event"]
    .astype("string")
    .str.strip()
    .fillna("<MISSING>")
    .replace("", "<EMPTY_STRING>")
)

tracking_event_summary = (
    event_display_values
    .value_counts(dropna=False)
    .rename_axis("event")
    .reset_index(name="frame_count")
)

tracking_event_summary["frame_percentage"] = (
    100
    * tracking_event_summary["frame_count"]
    / len(tracking_football_frames)
).round(3)

print(
    "Football frame records: "
    f"{len(tracking_football_frames):,}"
)
print(
    "Duplicated football-frame keys: "
    f"{duplicated_football_frames:,}"
)
print(
    "Missing event values: "
    f"{tracking_football_frames['event'].isna().sum():,}"
)
print(
    "Empty event strings: "
    f"{empty_event_count:,}"
)
print(
    "Event values with surrounding whitespace: "
    f"{event_whitespace_count:,}"
)

print("\nFrame-level event distribution:")
print(
    tracking_event_summary.to_string(
        index=False
    )
)

assert len(tracking_football_frames) == len(
    tracking_global_frame_counts
), "The football-frame count is inconsistent."

assert duplicated_football_frames == 0, (
    "Duplicated football records exist for a frame."
)

assert event_whitespace_count == 0, (
    "Event values contain surrounding whitespace."
)

print("\nFrame-level event inspection: PASSED")

Football frame records: 361,486
Duplicated football-frame keys: 0
Missing event values: 0
Empty event strings: 0
Event values with surrounding whitespace: 0

Frame-level event distribution:
                    event  frame_count  frame_percentage
                     None       333624            92.292
                ball_snap         8532              2.36
             pass_forward         7548             2.088
       autoevent_ballsnap         3767             1.042
    autoevent_passforward         3734             1.033
              play_action         1977             0.547
                      run          474             0.131
                  qb_sack          451             0.125
             pass_arrived          367             0.102
autoevent_passinterrupted          201             0.056
            man_in_motion          177             0.049
                 line_set          140             0.039
                    shift          133             0.037
            

### 2.21 Snap-event coverage

Manual and automatic snap markers are compared at the play level before
selecting a rule for identifying the snap frame.

In [73]:
MANUAL_SNAP_EVENT = "ball_snap"
AUTOMATIC_SNAP_EVENT = "autoevent_ballsnap"

SNAP_EVENT_VALUES = [
    MANUAL_SNAP_EVENT,
    AUTOMATIC_SNAP_EVENT,
]

snap_event_records = (
    tracking_football_frames.loc[
        tracking_football_frames[
            "event"
        ].isin(SNAP_EVENT_VALUES),
        [
            "gameId",
            "playId",
            "frameId",
            "event",
        ],
    ]
)

all_play_index = pd.MultiIndex.from_tuples(
    sorted(all_tracking_play_keys),
    names=PFF_PLAY_KEY,
)

snap_event_counts_by_play = (
    snap_event_records.groupby(
        [
            *PFF_PLAY_KEY,
            "event",
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        all_play_index,
        fill_value=0,
    )
)

for snap_event in SNAP_EVENT_VALUES:
    if snap_event not in (
        snap_event_counts_by_play.columns
    ):
        snap_event_counts_by_play[
            snap_event
        ] = 0

snap_event_counts_by_play = (
    snap_event_counts_by_play[
        SNAP_EVENT_VALUES
    ]
)

snap_presence_patterns = (
    snap_event_counts_by_play
    .value_counts()
    .rename("play_count")
    .reset_index()
    .sort_values(
        [
            MANUAL_SNAP_EVENT,
            AUTOMATIC_SNAP_EVENT,
        ],
        ascending=False,
        ignore_index=True,
    )
)

plays_without_snap_marker = int(
    snap_event_counts_by_play.sum(axis=1)
    .eq(0)
    .sum()
)

plays_with_multiple_manual_snaps = int(
    snap_event_counts_by_play[
        MANUAL_SNAP_EVENT
    ]
    .gt(1)
    .sum()
)

plays_with_multiple_automatic_snaps = int(
    snap_event_counts_by_play[
        AUTOMATIC_SNAP_EVENT
    ]
    .gt(1)
    .sum()
)

print("Snap-marker patterns by play:")
print(
    snap_presence_patterns.to_string(
        index=False
    )
)

print(
    "\nPlays without either snap marker: "
    f"{plays_without_snap_marker:,}"
)
print(
    "Plays with multiple manual snap markers: "
    f"{plays_with_multiple_manual_snaps:,}"
)
print(
    "Plays with multiple automatic snap markers: "
    f"{plays_with_multiple_automatic_snaps:,}"
)

Snap-marker patterns by play:
 ball_snap  autoevent_ballsnap  play_count
         1                   2          10
         1                   1        3746
         1                   0        4776
         0                   1           1
         0                   0          24

Plays without either snap marker: 24
Plays with multiple manual snap markers: 0
Plays with multiple automatic snap markers: 10


### 2.22 Snap-marker anomaly inspection

Plays with missing or duplicated snap markers are reviewed by week, source
file, frame count, and sequence of recorded events.

In [74]:
NO_EVENT_VALUE = "None"

snap_anomaly_summary = (
    snap_event_counts_by_play.reset_index()
)

snap_anomaly_summary = (
    snap_anomaly_summary.loc[
        (
            snap_anomaly_summary[
                MANUAL_SNAP_EVENT
            ] != 1
        )
        | (
            snap_anomaly_summary[
                AUTOMATIC_SNAP_EVENT
            ] > 1
        )
    ]
    .copy()
)

snap_anomaly_summary["anomaly_type"] = (
    "multiple_automatic"
)

snap_anomaly_summary.loc[
    (
        snap_anomaly_summary[
            MANUAL_SNAP_EVENT
        ] == 0
    )
    & (
        snap_anomaly_summary[
            AUTOMATIC_SNAP_EVENT
        ] == 1
    ),
    "anomaly_type",
] = "automatic_only"

snap_anomaly_summary.loc[
    (
        snap_anomaly_summary[
            MANUAL_SNAP_EVENT
        ] == 0
    )
    & (
        snap_anomaly_summary[
            AUTOMATIC_SNAP_EVENT
        ] == 0
    ),
    "anomaly_type",
] = "missing_both"

snap_anomaly_keys = snap_anomaly_summary[
    PFF_PLAY_KEY
]

snap_anomaly_frames = (
    tracking_football_frames.merge(
        snap_anomaly_keys,
        on=PFF_PLAY_KEY,
        how="inner",
        validate="many_to_one",
    )
)

recorded_anomaly_events = (
    snap_anomaly_frames.loc[
        snap_anomaly_frames["event"].ne(
            NO_EVENT_VALUE
        )
        & snap_anomaly_frames[
            "event"
        ].notna()
    ]
    .sort_values(
        [
            *PFF_PLAY_KEY,
            "frameId",
        ]
    )
    .copy()
)

recorded_anomaly_events[
    "event_at_frame"
] = (
    recorded_anomaly_events[
        "frameId"
    ].astype(str)
    + ":"
    + recorded_anomaly_events["event"].astype(str)
)

anomaly_event_sequences = (
    recorded_anomaly_events.groupby(
        PFF_PLAY_KEY,
        as_index=False,
    )
    .agg(
        event_sequence=(
            "event_at_frame",
            " | ".join,
        )
    )
)

anomaly_source_files = (
    snap_anomaly_frames.groupby(
        PFF_PLAY_KEY,
        as_index=False,
    )
    .agg(
        source_file_numbers=(
            "source_file_number",
            lambda values: sorted(set(values)),
        )
    )
)

snap_anomaly_summary = (
    snap_anomaly_summary.merge(
        tracking_play_frame_summary[
            [
                *PFF_PLAY_KEY,
                "frame_count",
            ]
        ],
        on=PFF_PLAY_KEY,
        how="left",
        validate="one_to_one",
    )
    .merge(
        tracking_game_week_map[
            [
                "gameId",
                "actual_week",
            ]
        ],
        on="gameId",
        how="left",
        validate="many_to_one",
    )
    .merge(
        anomaly_source_files,
        on=PFF_PLAY_KEY,
        how="left",
        validate="one_to_one",
    )
    .merge(
        anomaly_event_sequences,
        on=PFF_PLAY_KEY,
        how="left",
        validate="one_to_one",
    )
)

snap_anomaly_summary["event_sequence"] = (
    snap_anomaly_summary["event_sequence"]
    .fillna("<NO_RECORDED_EVENT>")
)

snap_anomaly_summary = (
    snap_anomaly_summary.sort_values(
        [
            "anomaly_type",
            "actual_week",
            "gameId",
            "playId",
        ],
        ignore_index=True,
    )
)

print(
    snap_anomaly_summary[
        [
            "anomaly_type",
            "actual_week",
            "source_file_numbers",
            "gameId",
            "playId",
            "frame_count",
            MANUAL_SNAP_EVENT,
            AUTOMATIC_SNAP_EVENT,
            "event_sequence",
        ]
    ].to_string(index=False)
)

      anomaly_type  actual_week source_file_numbers     gameId  playId  frame_count  ball_snap  autoevent_ballsnap                                                                                                                               event_sequence
    automatic_only            7                 [8] 2021102402    3191           35          0                   1                                                                                      1:autoevent_ballsnap | 11:play_action | 30:pass_forward
      missing_both            1                 [1] 2021091200    4367           33          0                   0                                                                                                                              28:pass_forward
      missing_both            1                 [1] 2021091202    3606           21          0                   0                                                                                                   16:pass_forward | 1

### 2.23 Manual-versus-automatic snap comparison

Automatic snap frames are compared with manual snap frames when both markers
occur exactly once within a play.

In [75]:
snap_frame_by_play = (
    snap_event_records.groupby(
        [
            *PFF_PLAY_KEY,
            "event",
        ]
    )["frameId"]
    .first()
    .unstack()
    .reindex(all_play_index)
)

single_manual_and_automatic_mask = (
    snap_event_counts_by_play[
        MANUAL_SNAP_EVENT
    ].eq(1)
    & snap_event_counts_by_play[
        AUTOMATIC_SNAP_EVENT
    ].eq(1)
)

single_snap_frame_comparison = (
    snap_frame_by_play.loc[
        single_manual_and_automatic_mask,
        SNAP_EVENT_VALUES,
    ]
    .rename(
        columns={
            MANUAL_SNAP_EVENT: (
                "manual_snap_frame"
            ),
            AUTOMATIC_SNAP_EVENT: (
                "automatic_snap_frame"
            ),
        }
    )
    .reset_index()
)

single_snap_frame_comparison[
    "automatic_minus_manual_frame"
] = (
    single_snap_frame_comparison[
        "automatic_snap_frame"
    ]
    - single_snap_frame_comparison[
        "manual_snap_frame"
    ]
)

snap_frame_offset_summary = (
    single_snap_frame_comparison[
        "automatic_minus_manual_frame"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

most_common_snap_offsets = (
    single_snap_frame_comparison[
        "automatic_minus_manual_frame"
    ]
    .value_counts()
    .rename_axis(
        "automatic_minus_manual_frame"
    )
    .reset_index(name="play_count")
    .head(15)
)

manual_snap_frame_summary = (
    snap_frame_by_play[
        MANUAL_SNAP_EVENT
    ]
    .dropna()
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

manual_selection_count = int(
    snap_event_counts_by_play[
        MANUAL_SNAP_EVENT
    ].eq(1).sum()
)

automatic_fallback_count = int(
    (
        snap_event_counts_by_play[
            MANUAL_SNAP_EVENT
        ].eq(0)
        & snap_event_counts_by_play[
            AUTOMATIC_SNAP_EVENT
        ].eq(1)
    ).sum()
)

unresolved_snap_count = int(
    snap_event_counts_by_play.sum(axis=1)
    .eq(0)
    .sum()
)

resolved_snap_count = (
    manual_selection_count
    + automatic_fallback_count
)

print(
    "Plays with one manual and one automatic marker: "
    f"{len(single_snap_frame_comparison):,}"
)

print("\nAutomatic-minus-manual frame offset:")
print(snap_frame_offset_summary.to_string())

print("\nMost common frame offsets:")
print(
    most_common_snap_offsets.to_string(
        index=False
    )
)

print("\nManual snap-frame distribution:")
print(manual_snap_frame_summary.to_string())

print("\nProposed snap coverage:")
print(
    f"Manual selection: {manual_selection_count:,}"
)
print(
    "Automatic fallback: "
    f"{automatic_fallback_count:,}"
)
print(
    f"Unresolved plays: {unresolved_snap_count:,}"
)
print(
    "Resolved coverage: "
    f"{resolved_snap_count / len(all_play_index):.2%}"
)

Plays with one manual and one automatic marker: 3,746

Automatic-minus-manual frame offset:
count    3746.000000
mean       -0.436466
std         6.336226
min      -167.000000
1%         -3.000000
5%         -2.000000
25%        -1.000000
50%        -1.000000
75%         1.000000
95%         2.000000
99%         4.000000
max         6.000000

Most common frame offsets:
 automatic_minus_manual_frame  play_count
                         -1.0        1943
                          1.0        1224
                          2.0         222
                         -2.0         177
                          3.0          57
                          4.0          45
                          5.0          22
                         -3.0          19
                         -4.0           7
                         -5.0           5
                          6.0           3
                         -6.0           3
                         -7.0           3
                        -85.0           

### 2.24 Final snap-quality rule

The primary analysis uses only explicit `ball_snap` events. Plays without a
manual snap marker are excluded because automatic markers contain occasional
large timing errors.

In [76]:
snap_quality_by_play = (
    snap_event_counts_by_play
    .reset_index()
    .merge(
        tracking_game_week_map[
            [
                "gameId",
                "actual_week",
            ]
        ],
        on="gameId",
        how="left",
        validate="many_to_one",
    )
)

snap_quality_by_play[
    "has_valid_manual_snap"
] = (
    snap_quality_by_play[
        MANUAL_SNAP_EVENT
    ].eq(1)
)

snap_quality_by_week = (
    snap_quality_by_play.groupby(
        "actual_week",
        as_index=False,
    )
    .agg(
        total_plays=(
            "playId",
            "size",
        ),
        valid_manual_snap_plays=(
            "has_valid_manual_snap",
            "sum",
        ),
    )
)

snap_quality_by_week[
    "excluded_plays"
] = (
    snap_quality_by_week["total_plays"]
    - snap_quality_by_week[
        "valid_manual_snap_plays"
    ]
)

snap_quality_by_week[
    "manual_snap_coverage_percentage"
] = (
    100
    * snap_quality_by_week[
        "valid_manual_snap_plays"
    ]
    / snap_quality_by_week["total_plays"]
).round(3)

manual_snap_frame_map = (
    tracking_football_frames.loc[
        tracking_football_frames[
            "event"
        ].eq(MANUAL_SNAP_EVENT),
        [
            "gameId",
            "playId",
            "frameId",
        ],
    ]
    .rename(
        columns={
            "frameId": "snap_frame_id",
        }
    )
    .merge(
        tracking_game_week_map[
            [
                "gameId",
                "actual_week",
            ]
        ],
        on="gameId",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        PFF_PLAY_KEY,
        ignore_index=True,
    )
)

snap_exclusion_play_keys = (
    snap_quality_by_play.loc[
        ~snap_quality_by_play[
            "has_valid_manual_snap"
        ],
        PFF_PLAY_KEY,
    ]
    .sort_values(
        PFF_PLAY_KEY,
        ignore_index=True,
    )
)

assert len(manual_snap_frame_map) == 8_532, (
    "The valid manual snap count is inconsistent."
)
assert not manual_snap_frame_map.duplicated(
    subset=PFF_PLAY_KEY
).any(), "Duplicated manual snap frames were detected."
assert len(snap_exclusion_play_keys) == 25, (
    "The snap-exclusion count is inconsistent."
)

print("Manual snap quality by actual week:")
print(
    snap_quality_by_week.to_string(
        index=False
    )
)

print(
    "\nValid manual snap plays: "
    f"{len(manual_snap_frame_map):,}"
)
print(
    "Excluded plays without manual snap: "
    f"{len(snap_exclusion_play_keys):,}"
)
print("Final manual-snap rule: PASSED")

Manual snap quality by actual week:
 actual_week  total_plays  valid_manual_snap_plays  excluded_plays  manual_snap_coverage_percentage
           1         1175                     1172               3                           99.745
           2         1067                     1062               5                           99.531
           3         1141                     1139               2                           99.825
           4         1113                     1108               5                           99.551
           5         1108                     1105               3                           99.729
           6         1004                     1002               2                           99.801
           7          917                      913               4                           99.564
           8         1032                     1031               1                           99.903

Valid manual snap plays: 8,532
Excluded plays without manual sn

### 2.25 Tracking numeric-range inspection

Numeric tracking variables are inspected through Parquet minimum and maximum
statistics to detect impossible coordinates, negative measurements, or invalid angles.

In [77]:
TRACKING_NUMERIC_RANGE_COLUMNS = [
    "frameId",
    "x",
    "y",
    "s",
    "a",
    "dis",
    "o",
    "dir",
]

tracking_numeric_range_records = []

for source_file_number, tracking_path in (
    tracking_path_by_file_number.items()
):
    parquet_file = pq.ParquetFile(
        tracking_path
    )
    file_metadata = parquet_file.metadata

    for row_group_index in range(
        file_metadata.num_row_groups
    ):
        row_group_metadata = (
            file_metadata.row_group(
                row_group_index
            )
        )

        for column_name in (
            TRACKING_NUMERIC_RANGE_COLUMNS
        ):
            column_index = (
                reference_schema.get_field_index(
                    column_name
                )
            )

            column_statistics = (
                row_group_metadata
                .column(column_index)
                .statistics
            )

            statistics_available = (
                column_statistics is not None
                and column_statistics.has_min_max
            )

            tracking_numeric_range_records.append(
                {
                    "source_file_number": (
                        source_file_number
                    ),
                    "column_name": column_name,
                    "minimum": (
                        column_statistics.min
                        if statistics_available
                        else None
                    ),
                    "maximum": (
                        column_statistics.max
                        if statistics_available
                        else None
                    ),
                    "null_count": (
                        column_statistics.null_count
                        if column_statistics
                        is not None
                        else None
                    ),
                    "statistics_available": (
                        statistics_available
                    ),
                }
            )

tracking_numeric_range_metadata = (
    pd.DataFrame(
        tracking_numeric_range_records
    )
)

all_range_statistics_available = (
    tracking_numeric_range_metadata[
        "statistics_available"
    ].all()
)

print(
    "Range statistics available for all "
    "numeric columns: "
    f"{all_range_statistics_available}"
)

if all_range_statistics_available:
    tracking_numeric_range_summary = (
        tracking_numeric_range_metadata.groupby(
            "column_name",
            sort=False,
            as_index=False,
        )
        .agg(
            global_minimum=("minimum", "min"),
            global_maximum=("maximum", "max"),
            missing_count=("null_count", "sum"),
        )
    )

    print("\nGlobal tracking numeric ranges:")
    print(
        tracking_numeric_range_summary.to_string(
            index=False
        )
    )
else:
    unavailable_range_statistics = (
        tracking_numeric_range_metadata.loc[
            ~tracking_numeric_range_metadata[
                "statistics_available"
            ],
            [
                "source_file_number",
                "column_name",
            ],
        ]
    )

    print("\nUnavailable range statistics:")
    print(
        unavailable_range_statistics.to_string(
            index=False
        )
    )

Range statistics available for all numeric columns: True

Global tracking numeric ranges:
column_name  global_minimum  global_maximum  missing_count
    frameId            1.00          203.00              0
          x           -3.55          121.15              0
          y           -4.53           57.72              0
          s            0.00           29.34              0
          a            0.00           50.69              0
        dis            0.00           10.45              0
          o            0.00          360.00         361486
        dir            0.00          360.00         361486


### 2.26 Numeric ranges by entity type

Numeric ranges are separated between player and football records because their
movement characteristics and applicable fields differ substantially.

In [78]:
FIELD_X_MIN = 0.0
FIELD_X_MAX = 120.0
FIELD_Y_MIN = 0.0
FIELD_Y_MAX = 160.0 / 3.0

TRACKING_ENTITY_RANGE_COLUMNS = [
    "team",
    "x",
    "y",
    "s",
    "a",
    "dis",
    "o",
    "dir",
]

TRACKING_MEASUREMENT_COLUMNS = [
    "x",
    "y",
    "s",
    "a",
    "dis",
    "o",
    "dir",
]

entity_range_accumulators = {}

for entity_type in [
    "player",
    "football",
]:
    accumulator = {
        "entity_type": entity_type,
        "rows": 0,
        "x_outside_field": 0,
        "y_outside_field": 0,
    }

    for column_name in (
        TRACKING_MEASUREMENT_COLUMNS
    ):
        accumulator[
            f"{column_name}_minimum"
        ] = None
        accumulator[
            f"{column_name}_maximum"
        ] = None

    entity_range_accumulators[
        entity_type
    ] = accumulator

for tracking_path in tracking_paths:
    parquet_file = pq.ParquetFile(
        tracking_path
    )

    for record_batch in parquet_file.iter_batches(
        batch_size=TRACKING_BATCH_SIZE,
        columns=TRACKING_ENTITY_RANGE_COLUMNS,
    ):
        batch_data = record_batch.to_pandas()

        football_mask = batch_data["team"].eq(
            FOOTBALL_TEAM_VALUE
        )

        entity_masks = {
            "player": ~football_mask,
            "football": football_mask,
        }

        for entity_type, entity_mask in (
            entity_masks.items()
        ):
            entity_data = batch_data.loc[
                entity_mask
            ]

            accumulator = (
                entity_range_accumulators[
                    entity_type
                ]
            )

            accumulator["rows"] += len(
                entity_data
            )

            accumulator[
                "x_outside_field"
            ] += int(
                (
                    entity_data["x"].lt(
                        FIELD_X_MIN
                    )
                    | entity_data["x"].gt(
                        FIELD_X_MAX
                    )
                ).sum()
            )

            accumulator[
                "y_outside_field"
            ] += int(
                (
                    entity_data["y"].lt(
                        FIELD_Y_MIN
                    )
                    | entity_data["y"].gt(
                        FIELD_Y_MAX
                    )
                ).sum()
            )

            for column_name in (
                TRACKING_MEASUREMENT_COLUMNS
            ):
                non_missing_values = (
                    entity_data[column_name]
                    .dropna()
                )

                if non_missing_values.empty:
                    continue

                batch_minimum = float(
                    non_missing_values.min()
                )
                batch_maximum = float(
                    non_missing_values.max()
                )

                current_minimum = accumulator[
                    f"{column_name}_minimum"
                ]
                current_maximum = accumulator[
                    f"{column_name}_maximum"
                ]

                accumulator[
                    f"{column_name}_minimum"
                ] = (
                    batch_minimum
                    if current_minimum is None
                    else min(
                        current_minimum,
                        batch_minimum,
                    )
                )

                accumulator[
                    f"{column_name}_maximum"
                ] = (
                    batch_maximum
                    if current_maximum is None
                    else max(
                        current_maximum,
                        batch_maximum,
                    )
                )

tracking_entity_range_summary = pd.DataFrame(
    entity_range_accumulators.values()
)

tracking_boundary_summary = (
    tracking_entity_range_summary[
        [
            "entity_type",
            "rows",
            "x_outside_field",
            "y_outside_field",
        ]
    ]
    .copy()
)

tracking_boundary_summary[
    "x_outside_percentage"
] = (
    100
    * tracking_boundary_summary[
        "x_outside_field"
    ]
    / tracking_boundary_summary["rows"]
).round(4)

tracking_boundary_summary[
    "y_outside_percentage"
] = (
    100
    * tracking_boundary_summary[
        "y_outside_field"
    ]
    / tracking_boundary_summary["rows"]
).round(4)

print("Numeric ranges by entity type:")
print(
    tracking_entity_range_summary.to_string(
        index=False
    )
)

print("\nField-boundary violations:")
print(
    tracking_boundary_summary.to_string(
        index=False
    )
)

Numeric ranges by entity type:
entity_type    rows  x_outside_field  y_outside_field  x_minimum  x_maximum  y_minimum  y_maximum  s_minimum  s_maximum  a_minimum  a_maximum  dis_minimum  dis_maximum  o_minimum  o_maximum  dir_minimum  dir_maximum
     player 7952692              112              828      -3.55     121.15      -4.21      56.59        0.0      11.18        0.0      21.92          0.0         3.45        0.0      360.0          0.0        360.0
   football  361486                0              204       2.76     119.20      -4.53      57.72        0.0      29.34        0.0      50.69          0.0        10.45        NaN        NaN          NaN          NaN

Field-boundary violations:
entity_type    rows  x_outside_field  y_outside_field  x_outside_percentage  y_outside_percentage
     player 7952692              112              828                0.0014                0.0104
   football  361486                0              204                0.0000                0.0564

### 2.27 Play-direction and timestamp validation

Football records are used to verify that play direction remains constant within
each play and that tracking timestamps advance in regular 0.1-second intervals.

In [86]:
FOOTBALL_SEQUENCE_COLUMNS = [
    "gameId",
    "playId",
    "frameId",
    "team",
    "playDirection",
    "time",
]

football_sequence_parts = []

for source_file_number, tracking_path in (
    tracking_path_by_file_number.items()
):
    football_sequence_table = pq.read_table(
        tracking_path,
        columns=FOOTBALL_SEQUENCE_COLUMNS,
        filters=[
            (
                "team",
                "=",
                FOOTBALL_TEAM_VALUE,
            )
        ],
    )

    football_sequence_part = (
        football_sequence_table.to_pandas()
    )
    football_sequence_part[
        "source_file_number"
    ] = source_file_number

    football_sequence_parts.append(
        football_sequence_part
    )

tracking_football_sequence = pd.concat(
    football_sequence_parts,
    ignore_index=True,
)

tracking_football_sequence[
    "parsed_time"
] = pd.to_datetime(
    tracking_football_sequence["time"],
    errors="coerce",
    utc=True,
)

unparseable_time_count = int(
    tracking_football_sequence[
        "parsed_time"
    ].isna().sum()
)

observed_play_directions = sorted(
    tracking_football_sequence[
        "playDirection"
    ]
    .dropna()
    .unique()
    .tolist()
)

play_direction_by_play = (
    tracking_football_sequence.groupby(
        PFF_PLAY_KEY,
        as_index=False,
    )
    .agg(
        direction_count=(
            "playDirection",
            "nunique",
        ),
        play_direction=(
            "playDirection",
            "first",
        ),
    )
)

inconsistent_direction_plays = (
    play_direction_by_play.loc[
        play_direction_by_play[
            "direction_count"
        ] != 1
    ]
)

play_direction_summary = (
    play_direction_by_play[
        "play_direction"
    ]
    .value_counts()
    .rename_axis("play_direction")
    .reset_index(name="play_count")
)

tracking_football_sequence = (
    tracking_football_sequence.sort_values(
        [
            *PFF_PLAY_KEY,
            "frameId",
        ],
        ignore_index=True,
    )
)

tracking_football_sequence[
    "time_step_seconds"
] = (
    tracking_football_sequence.groupby(
        PFF_PLAY_KEY
    )["parsed_time"]
    .diff()
    .dt.total_seconds()
)

valid_time_steps = (
    tracking_football_sequence[
        "time_step_seconds"
    ]
    .dropna()
)

rounded_time_steps = valid_time_steps.round(6)

time_step_distribution = (
    rounded_time_steps
    .value_counts()
    .sort_index()
    .rename_axis("time_step_seconds")
    .reset_index(name="frame_transitions")
)

unexpected_time_step_count = int(
    rounded_time_steps.ne(0.1).sum()
)

print(
    "Observed playDirection values: "
    f"{observed_play_directions}"
)
print(
    "Unparseable timestamps: "
    f"{unparseable_time_count:,}"
)
print(
    "Plays with inconsistent direction: "
    f"{len(inconsistent_direction_plays):,}"
)

print("\nPlay-direction distribution:")
print(
    play_direction_summary.to_string(
        index=False
    )
)

print("\nTimestamp-step distribution:")
print(
    time_step_distribution.to_string(
        index=False
    )
)

assert observed_play_directions == [
    "left",
    "right",
], "Unexpected playDirection values were detected."

assert unparseable_time_count == 0, (
    "Unparseable tracking timestamps were detected."
)

assert inconsistent_direction_plays.empty, (
    "playDirection changes within at least one play."
)
print(
    "\nTimestamp transitions requiring review: "
    f"{unexpected_time_step_count:,}"
)

print(
    "Play-direction and timestamp inspection: COMPLETED"
)


Observed playDirection values: ['left', 'right']
Unparseable timestamps: 0
Plays with inconsistent direction: 0

Play-direction distribution:
play_direction  play_count
          left        4431
         right        4126

Timestamp-step distribution:
 time_step_seconds  frame_transitions
               0.1             352923
               8.1                  1
               9.5                  1
              10.0                  1
              10.5                  1
              13.3                  2

Timestamp transitions requiring review: 6
Play-direction and timestamp inspection: COMPLETED


In [83]:
tracking_football_sequence = (
    tracking_football_sequence.merge(
        tracking_football_frames[
            [
                *TRACKING_FRAME_KEY,
                "event",
            ]
        ],
        on=TRACKING_FRAME_KEY,
        how="left",
        validate="one_to_one",
    )
)

sequence_groups = (
    tracking_football_sequence.groupby(
        PFF_PLAY_KEY,
        sort=False,
    )
)

tracking_football_sequence[
    "previous_frame_id"
] = sequence_groups["frameId"].shift(1)

tracking_football_sequence[
    "previous_time"
] = sequence_groups["time"].shift(1)

tracking_football_sequence[
    "previous_event"
] = sequence_groups["event"].shift(1)

unexpected_time_transitions = (
    tracking_football_sequence.loc[
        tracking_football_sequence[
            "time_step_seconds"
        ].notna()
        & tracking_football_sequence[
            "time_step_seconds"
        ].round(6).ne(0.1)
    ]
    .merge(
        manual_snap_frame_map[
            [
                *PFF_PLAY_KEY,
                "snap_frame_id",
            ]
        ],
        on=PFF_PLAY_KEY,
        how="left",
        validate="many_to_one",
    )
    .merge(
        tracking_game_week_map[
            [
                "gameId",
                "actual_week",
            ]
        ],
        on="gameId",
        how="left",
        validate="many_to_one",
    )
)

unexpected_time_transitions[
    "gap_crosses_snap"
] = (
    unexpected_time_transitions[
        "snap_frame_id"
    ].notna()
    & (
        unexpected_time_transitions[
            "previous_frame_id"
        ]
        < unexpected_time_transitions[
            "snap_frame_id"
        ]
    )
    & (
        unexpected_time_transitions[
            "frameId"
        ]
        >= unexpected_time_transitions[
            "snap_frame_id"
        ]
    )
)

print(
    unexpected_time_transitions[
        [
            "actual_week",
            "source_file_number",
            "gameId",
            "playId",
            "snap_frame_id",
            "previous_frame_id",
            "frameId",
            "previous_time",
            "time",
            "time_step_seconds",
            "previous_event",
            "event",
            "gap_crosses_snap",
        ]
    ].to_string(index=False)
)

 actual_week  source_file_number     gameId  playId  snap_frame_id  previous_frame_id  frameId           previous_time                    time  time_step_seconds previous_event event  gap_crosses_snap
           1                   1 2021091213    3284              6                2.0        3 2021-09-13T02:55:14.200 2021-09-13T02:55:24.700               10.5           None  None             False
           2                   2 2021091901    1274              6                3.0        4 2021-09-19T17:51:48.100 2021-09-19T17:52:01.400               13.3           None  None             False
           3                   3 2021092605    1069              7                2.0        3 2021-09-26T17:46:34.200 2021-09-26T17:46:42.300                8.1           None  None             False
           5                   6 2021101001    2821              6                3.0        4 2021-10-10T19:12:32.300 2021-10-10T19:12:42.300               10.0           None  None             F

In [85]:
assert unexpected_time_transitions[
    "snap_frame_id"
].notna().all(), (
    "A timestamp gap belongs to a play "
    "without a valid manual snap."
)

assert (
    unexpected_time_transitions["frameId"]
    < unexpected_time_transitions[
        "snap_frame_id"
    ]
).all(), (
    "A timestamp gap reaches or occurs after the snap."
)

print(
    "Timestamp-gap validation: PASSED "
    "(6 documented pre-snap gaps)"
)

Timestamp-gap validation: PASSED (6 documented pre-snap gaps)


### 2.28 PFF-to-tracking player-play coverage

Unique player-play keys are extracted from tracking and compared one to one
against the PFF scouting table.

In [87]:
TRACKING_PLAYER_PLAY_COLUMNS = [
    "gameId",
    "playId",
    "nflId",
    "team",
]

tracking_player_play_parts = []

for tracking_path in tracking_paths:
    parquet_file = pq.ParquetFile(
        tracking_path
    )

    for record_batch in parquet_file.iter_batches(
        batch_size=TRACKING_BATCH_SIZE,
        columns=TRACKING_PLAYER_PLAY_COLUMNS,
    ):
        batch_data = record_batch.to_pandas()

        batch_player_plays = (
            batch_data.loc[
                batch_data["nflId"].notna(),
                TRACKING_PLAYER_PLAY_COLUMNS,
            ]
            .drop_duplicates()
        )

        tracking_player_play_parts.append(
            batch_player_plays
        )

tracking_player_play_candidates = pd.concat(
    tracking_player_play_parts,
    ignore_index=True,
)

tracking_player_play_candidates[
    "nflId"
] = (
    tracking_player_play_candidates[
        "nflId"
    ].astype("int64")
)

team_count_per_player_play = (
    tracking_player_play_candidates.groupby(
        PFF_PLAYER_PLAY_KEY
    )["team"]
    .nunique(dropna=False)
)

conflicting_team_assignments = int(
    team_count_per_player_play.gt(1).sum()
)

tracking_player_plays = (
    tracking_player_play_candidates
    .drop_duplicates(
        subset=PFF_PLAYER_PLAY_KEY
    )
    .sort_values(
        PFF_PLAYER_PLAY_KEY,
        ignore_index=True,
    )
)

tracking_players_per_play = (
    tracking_player_plays.groupby(
        PFF_PLAY_KEY
    )
    .size()
)

tracking_players_per_play_distribution = (
    tracking_players_per_play
    .value_counts()
    .sort_index()
    .rename_axis("players_per_play")
    .reset_index(name="play_count")
)

pff_player_play_keys = (
    pff_data[PFF_PLAYER_PLAY_KEY]
    .drop_duplicates()
)

pff_tracking_key_comparison = (
    pff_player_play_keys.merge(
        tracking_player_plays[
            PFF_PLAYER_PLAY_KEY
        ],
        on=PFF_PLAYER_PLAY_KEY,
        how="outer",
        indicator=True,
        validate="one_to_one",
    )
)

pff_tracking_coverage_summary = (
    pff_tracking_key_comparison[
        "_merge"
    ]
    .value_counts()
    .reindex(
        [
            "both",
            "left_only",
            "right_only",
        ],
        fill_value=0,
    )
    .rename_axis("coverage_status")
    .reset_index(name="player_play_records")
)

print(
    "Unique tracking player-play records: "
    f"{len(tracking_player_plays):,}"
)
print(
    "Conflicting team assignments: "
    f"{conflicting_team_assignments:,}"
)

print("\nTracking players per play:")
print(
    tracking_players_per_play_distribution.to_string(
        index=False
    )
)

print("\nPFF-to-tracking key coverage:")
print(
    pff_tracking_coverage_summary.to_string(
        index=False
    )
)

assert conflicting_team_assignments == 0, (
    "At least one player-play has conflicting teams."
)

assert (
    tracking_players_per_play_distribution[
        "players_per_play"
    ].tolist()
    == [22]
), "Tracking does not contain exactly 22 players per play."

assert (
    pff_tracking_coverage_summary.set_index(
        "coverage_status"
    ).loc["left_only", "player_play_records"]
    == 0
), "Some PFF player-play keys are absent from tracking."

assert (
    pff_tracking_coverage_summary.set_index(
        "coverage_status"
    ).loc["right_only", "player_play_records"]
    == 0
), "Some tracking player-play keys are absent from PFF."

print("\nPFF-to-tracking key validation: PASSED")

Unique tracking player-play records: 188,254
Conflicting team assignments: 0

Tracking players per play:
 players_per_play  play_count
               22        8557

PFF-to-tracking key coverage:
coverage_status  player_play_records
           both               188254
      left_only                    0
     right_only                    0

PFF-to-tracking key validation: PASSED


### 2.29 Manual-snap exclusion impact

The impact of excluding plays without a manual snap marker is quantified by
PFF role before constructing the analytical dataset.

In [88]:
pff_with_snap_quality = (
    pff_data.merge(
        snap_quality_by_play[
            [
                "gameId",
                "playId",
                "actual_week",
                "has_valid_manual_snap",
            ]
        ],
        on=PFF_PLAY_KEY,
        how="left",
        validate="many_to_one",
    )
)

missing_snap_quality_assignments = int(
    pff_with_snap_quality[
        "has_valid_manual_snap"
    ].isna().sum()
)

snap_impact_by_role = (
    pff_with_snap_quality.groupby(
        PFF_ROLE_COLUMN,
        as_index=False,
    )
    .agg(
        total_player_plays=(
            "nflId",
            "size",
        ),
        retained_player_plays=(
            "has_valid_manual_snap",
            "sum",
        ),
    )
)

snap_impact_by_role[
    "excluded_player_plays"
] = (
    snap_impact_by_role[
        "total_player_plays"
    ]
    - snap_impact_by_role[
        "retained_player_plays"
    ]
)

snap_impact_by_role[
    "retained_percentage"
] = (
    100
    * snap_impact_by_role[
        "retained_player_plays"
    ]
    / snap_impact_by_role[
        "total_player_plays"
    ]
).round(3)

snap_impact_by_role = (
    snap_impact_by_role.sort_values(
        PFF_ROLE_COLUMN,
        ignore_index=True,
    )
)

total_excluded_player_plays = int(
    (
        ~pff_with_snap_quality[
            "has_valid_manual_snap"
        ]
    ).sum()
)

print("Manual-snap exclusion impact by role:")
print(
    snap_impact_by_role.to_string(
        index=False
    )
)

print(
    "\nTotal excluded player-play records: "
    f"{total_excluded_player_plays:,}"
)
print(
    "Missing snap-quality assignments: "
    f"{missing_snap_quality_assignments:,}"
)

assert missing_snap_quality_assignments == 0, (
    "Some PFF plays lack a snap-quality assignment."
)

assert total_excluded_player_plays == (
    len(snap_exclusion_play_keys) * 22
), (
    "Excluded player-play count is inconsistent "
    "with 22 players per excluded play."
)

print("\nManual-snap exclusion impact: PASSED")

Manual-snap exclusion impact by role:
  pff_role  total_player_plays  retained_player_plays  excluded_player_plays  retained_percentage
  Coverage               57765                  57593                    172               99.702
      Pass                8557                   8532                     25               99.708
Pass Block               46057                  45927                    130               99.718
Pass Route               39513                  39393                    120               99.696
 Pass Rush               36362                  36259                    103               99.717

Total excluded player-play records: 550
Missing snap-quality assignments: 0

Manual-snap exclusion impact: PASSED


### 2.30 Consolidated data-quality report

All structural validations, documented exceptions, and methodological
decisions are consolidated into a version-controlled quality report.

In [89]:
manual_snap_coverage_percentage = (
    100
    * len(manual_snap_frame_map)
    / len(all_tracking_play_keys)
)

pass_rush_impact = (
    snap_impact_by_role.loc[
        snap_impact_by_role[
            PFF_ROLE_COLUMN
        ].eq(PASS_RUSH_ROLE)
    ]
    .iloc[0]
)

coverage_lookup = (
    pff_tracking_coverage_summary
    .set_index("coverage_status")[
        "player_play_records"
    ]
)

player_boundary_result = (
    tracking_boundary_summary.loc[
        tracking_boundary_summary[
            "entity_type"
        ].eq("player")
    ]
    .iloc[0]
)

data_quality_records = [
    {
        "check_id": "DQ-01",
        "area": "Raw files",
        "status": "PASS",
        "result": (
            f"{len(data_file_inventory)} data files; "
            f"{total_raw_size_gib:.3f} GiB; "
            "no missing, unexpected, or empty files"
        ),
        "decision": (
            "Preserve raw files unchanged"
        ),
    },
    {
        "check_id": "DQ-02",
        "area": "Parquet structure",
        "status": "PASS",
        "result": (
            f"{tracking_metadata['rows'].sum():,} rows; "
            "16 columns; identical schemas"
        ),
        "decision": (
            "Weekly files can be concatenated"
        ),
    },
    {
        "check_id": "DQ-03",
        "area": "Temporal partition",
        "status": "PASS_WITH_NOTE",
        "result": (
            "Physical files do not correspond "
            "one-to-one with actual NFL weeks"
        ),
        "decision": (
            "Assign actual_week from gameId date; "
            "never split by Parquet filename"
        ),
    },
    {
        "check_id": "DQ-04",
        "area": "PFF key integrity",
        "status": "PASS",
        "result": (
            f"{len(pff_data):,} player-play rows; "
            f"{unique_plays:,} plays; "
            "no missing or duplicated keys"
        ),
        "decision": (
            "Use gameId + playId + nflId "
            "as the player-play key"
        ),
    },
    {
        "check_id": "DQ-05",
        "area": "PFF missingness",
        "status": "PASS_WITH_NOTE",
        "result": (
            "Missingness is structural by PFF role; "
            "Pass Rush hit/hurry/sack fields are complete"
        ),
        "decision": (
            "Do not impute PFF missing values globally"
        ),
    },
    {
        "check_id": "DQ-06",
        "area": "PFF value domains",
        "status": "PASS",
        "result": (
            "All indicator fields contain only "
            "0, 1, or missing; categorical strings are clean"
        ),
        "decision": (
            "Retain raw categorical values for later analysis"
        ),
    },
    {
        "check_id": "DQ-07",
        "area": "Team metadata",
        "status": "PASS_WITH_NOTE",
        "result": (
            "32 tracking teams covered; unused aliases: "
            f"{', '.join(unused_team_metadata_values)}"
        ),
        "decision": (
            "Retain historical aliases in reference metadata"
        ),
    },
    {
        "check_id": "DQ-08",
        "area": "Tracking entity identity",
        "status": "PASS",
        "result": (
            f"{len(tracking_global_frame_counts):,} frames; "
            "missing identity/orientation occurs only for football"
        ),
        "decision": (
            "Treat football records separately from players"
        ),
    },
    {
        "check_id": "DQ-09",
        "area": "Tracking frame integrity",
        "status": "PASS",
        "result": (
            "Exactly 23 entities per frame; "
            "no duplicated entity-frame keys or frame gaps"
        ),
        "decision": (
            "Use gameId + playId + frameId + entity "
            "as tracking grain"
        ),
    },
    {
        "check_id": "DQ-10",
        "area": "Manual snap coverage",
        "status": "PASS_WITH_NOTE",
        "result": (
            f"{len(manual_snap_frame_map):,} valid plays "
            f"({manual_snap_coverage_percentage:.3f}%); "
            f"{len(snap_exclusion_play_keys)} excluded plays"
        ),
        "decision": (
            "Use only explicit ball_snap; "
            "exclude plays without a manual marker"
        ),
    },
    {
        "check_id": "DQ-11",
        "area": "Tracking time",
        "status": "PASS_WITH_NOTE",
        "result": (
            f"{len(unexpected_time_transitions)} timestamp "
            "gaps, all occurring before the snap"
        ),
        "decision": (
            "Do not assume uniform pre-snap timing; "
            "snap and post-snap sequence remain valid"
        ),
    },
    {
        "check_id": "DQ-12",
        "area": "Tracking numeric ranges",
        "status": "PASS_WITH_NOTE",
        "result": (
            "Rare player boundary observations: "
            f"x={int(player_boundary_result['x_outside_field'])}, "
            f"y={int(player_boundary_result['y_outside_field'])}"
        ),
        "decision": (
            "Do not remove or cap globally; "
            "reassess at the snap snapshot"
        ),
    },
    {
        "check_id": "DQ-13",
        "area": "PFF-tracking coverage",
        "status": "PASS",
        "result": (
            f"{int(coverage_lookup['both']):,} matched keys; "
            f"{int(coverage_lookup['left_only'])} PFF-only; "
            f"{int(coverage_lookup['right_only'])} tracking-only"
        ),
        "decision": (
            "Proceed with a validated one-to-one "
            "player-play relationship"
        ),
    },
    {
        "check_id": "DQ-14",
        "area": "Snap exclusion impact",
        "status": "PASS_WITH_NOTE",
        "result": (
            f"{int(pass_rush_impact['retained_player_plays']):,} "
            "Pass Rush records retained; "
            f"{int(pass_rush_impact['excluded_player_plays'])} excluded"
        ),
        "decision": (
            "Document the 0.283% Pass Rush reduction"
        ),
    },
]

data_quality_report = pd.DataFrame(
    data_quality_records
)

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DATA_QUALITY_REPORT_PATH = (
    REPORTS_DIR / "data_quality_report.csv"
)

data_quality_report.to_csv(
    DATA_QUALITY_REPORT_PATH,
    index=False,
    encoding="utf-8",
)

assert DATA_QUALITY_REPORT_PATH.exists(), (
    "The data-quality report was not created."
)

print(data_quality_report.to_string(index=False))

print(
    "\nData-quality report saved to: "
    f"{DATA_QUALITY_REPORT_PATH}"
)
print("Consolidated data-quality report: PASSED")

check_id                     area         status                                                                              result                                                                        decision
   DQ-01                Raw files           PASS                    10 data files; 0.240 GiB; no missing, unexpected, or empty files                                                    Preserve raw files unchanged
   DQ-02        Parquet structure           PASS                                       8,314,178 rows; 16 columns; identical schemas                                                Weekly files can be concatenated
   DQ-03       Temporal partition PASS_WITH_NOTE                   Physical files do not correspond one-to-one with actual NFL weeks            Assign actual_week from gameId date; never split by Parquet filename
   DQ-04        PFF key integrity           PASS                188,254 player-play rows; 8,557 plays; no missing or duplicated keys                

# Stage 3 — Pressure Target Definition and Validation

In [90]:
# Stage 3.1 — Build the valid Pass Rush universe

PASS_RUSH_ROLE = "Pass Rush"
EXPECTED_VALID_PASS_RUSH_ROWS = 36_259

required_columns = {
    "gameId",
    "playId",
    "nflId",
    "actual_week",
    "has_valid_manual_snap",
    "pff_role",
    "pff_hit",
    "pff_hurry",
    "pff_sack",
}

missing_columns = required_columns.difference(
    pff_with_snap_quality.columns
)

assert not missing_columns, (
    f"Missing required columns: {sorted(missing_columns)}"
)

valid_pass_rush_mask = (
    pff_with_snap_quality["pff_role"].eq(PASS_RUSH_ROLE)
    & pff_with_snap_quality["has_valid_manual_snap"].eq(True)
)

pass_rush_universe = (
    pff_with_snap_quality.loc[valid_pass_rush_mask]
    .copy()
)

assert len(pass_rush_universe) == EXPECTED_VALID_PASS_RUSH_ROWS, (
    "Unexpected number of valid Pass Rush rows: "
    f"{len(pass_rush_universe):,}"
)

pass_rush_universe_summary = pd.Series(
    {
        "valid_pass_rush_rows": len(pass_rush_universe),
        "unique_plays": (
            pass_rush_universe[["gameId", "playId"]]
            .drop_duplicates()
            .shape[0]
        ),
        "unique_players": pass_rush_universe["nflId"].nunique(),
        "actual_weeks": pass_rush_universe["actual_week"].nunique(),
    },
    name="value",
)

pass_rush_universe_summary

valid_pass_rush_rows    36259
unique_plays             8531
unique_players            698
actual_weeks                8
Name: value, dtype: int64

In [91]:
# Stage 3.2 — Identify valid plays without a Pass Rush record

EXPECTED_VALID_PLAY_COUNT = 8_532
EXPECTED_PASS_RUSH_PLAY_COUNT = 8_531

valid_play_keys = (
    pff_with_snap_quality.loc[
        pff_with_snap_quality["has_valid_manual_snap"].eq(True),
        ["gameId", "playId", "actual_week"],
    ]
    .drop_duplicates()
)

pass_rush_play_keys = (
    pass_rush_universe[["gameId", "playId"]]
    .drop_duplicates()
)

assert len(valid_play_keys) == EXPECTED_VALID_PLAY_COUNT
assert len(pass_rush_play_keys) == EXPECTED_PASS_RUSH_PLAY_COUNT

plays_without_pass_rush = (
    valid_play_keys.merge(
        pass_rush_play_keys,
        on=["gameId", "playId"],
        how="left",
        indicator=True,
    )
    .loc[
        lambda data: data["_merge"].eq("left_only"),
        ["actual_week", "gameId", "playId"],
    ]
    .sort_values(["actual_week", "gameId", "playId"])
    .reset_index(drop=True)
)

assert len(plays_without_pass_rush) == 1, (
    "Expected exactly one valid play without Pass Rush, "
    f"but found {len(plays_without_pass_rush):,}."
)

plays_without_pass_rush

,actual_week,gameId,playId
0,6,2021101700,2372


In [92]:
# Stage 3.3 — Inspect roles in the play without Pass Rush

missing_game_id = int(
    plays_without_pass_rush.loc[0, "gameId"]
)
missing_play_id = int(
    plays_without_pass_rush.loc[0, "playId"]
)

missing_play_mask = (
    pff_with_snap_quality["gameId"].eq(missing_game_id)
    & pff_with_snap_quality["playId"].eq(missing_play_id)
)

play_without_pass_rush_role_summary = (
    pff_with_snap_quality.loc[missing_play_mask, "pff_role"]
    .value_counts()
    .rename_axis("pff_role")
    .reset_index(name="player_count")
)

assert play_without_pass_rush_role_summary[
    "player_count"
].sum() == 22

assert not play_without_pass_rush_role_summary[
    "pff_role"
].eq(PASS_RUSH_ROLE).any()

play_without_pass_rush_role_summary

,pff_role,player_count
0,Coverage,11
1,Pass Block,6
2,Pass Route,4
3,Pass,1


In [93]:
# Stage 3.4 — Validate pressure component quality

PRESSURE_COMPONENT_COLUMNS = [
    "pff_hurry",
    "pff_hit",
    "pff_sack",
]

pressure_component_quality = pd.DataFrame(
    {
        "non_missing": (
            pass_rush_universe[PRESSURE_COMPONENT_COLUMNS]
            .notna()
            .sum()
        ),
        "missing": (
            pass_rush_universe[PRESSURE_COMPONENT_COLUMNS]
            .isna()
            .sum()
        ),
        "zeros": (
            pass_rush_universe[PRESSURE_COMPONENT_COLUMNS]
            .eq(0)
            .sum()
        ),
        "ones": (
            pass_rush_universe[PRESSURE_COMPONENT_COLUMNS]
            .eq(1)
            .sum()
        ),
    }
).rename_axis("component").reset_index()

assert pressure_component_quality["missing"].eq(0).all(), (
    "Pressure components contain missing values."
)

assert (
    pass_rush_universe[PRESSURE_COMPONENT_COLUMNS]
    .isin([0, 1])
    .all()
    .all()
), "Pressure components contain values other than 0 or 1."

pressure_component_quality

,component,non_missing,missing,zeros,ones
0,pff_hurry,36259,0,33450,2809
1,pff_hit,36259,0,35438,821
2,pff_sack,36259,0,35675,584


In [94]:
# Stage 3.5 — Examine pressure component combinations

pressure_component_combinations = (
    pass_rush_universe
    .groupby(
        PRESSURE_COMPONENT_COLUMNS,
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
)

pressure_component_combinations["active_components"] = (
    pressure_component_combinations[
        PRESSURE_COMPONENT_COLUMNS
    ]
    .sum(axis=1)
    .astype("int64")
)

pressure_component_combinations["percentage"] = (
    pressure_component_combinations["rows"]
    .div(len(pass_rush_universe))
    .mul(100)
    .round(3)
)

pressure_component_combinations = (
    pressure_component_combinations[
        [
            *PRESSURE_COMPONENT_COLUMNS,
            "active_components",
            "rows",
            "percentage",
        ]
    ]
    .sort_values(
        ["active_components", "rows"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

assert pressure_component_combinations["rows"].sum() == len(
    pass_rush_universe
)

pressure_component_combinations

,pff_hurry,pff_hit,pff_sack,active_components,rows,percentage
0,0.0,0.0,0.0,0,32045,88.378
1,1.0,0.0,0.0,1,2809,7.747
2,0.0,1.0,0.0,1,821,2.264
3,0.0,0.0,1.0,1,584,1.611


In [96]:
# Stage 3.6 — Create the binary pressure target

TARGET_COLUMN = "pressure"
EXPECTED_PRESSURE_NEGATIVES = 32_045
EXPECTED_PRESSURE_POSITIVES = 4_214

pressure_data = pass_rush_universe.copy()

pressure_data[TARGET_COLUMN] = (
    pressure_data[PRESSURE_COMPONENT_COLUMNS]
    .eq(1)
    .any(axis=1)
    .astype("int8")
)

pressure_counts = pressure_data[TARGET_COLUMN].value_counts()

assert not pressure_data[TARGET_COLUMN].isna().any()
assert set(pressure_data[TARGET_COLUMN].unique()) == {0, 1}
assert pressure_counts.loc[0] == EXPECTED_PRESSURE_NEGATIVES
assert pressure_counts.loc[1] == EXPECTED_PRESSURE_POSITIVES

pressure_target_summary = (
    pressure_data[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis(TARGET_COLUMN)
    .reset_index(name="rows")
)

pressure_target_summary["outcome"] = (
    pressure_target_summary[TARGET_COLUMN]
    .map(
        {
            0: "No pressure",
            1: "Pressure",
        }
    )
)

pressure_target_summary["percentage"] = (
    pressure_target_summary["rows"]
    .div(len(pressure_data))
    .mul(100)
    .round(3)
)

pressure_target_summary = pressure_target_summary[
    [
        TARGET_COLUMN,
        "outcome",
        "rows",
        "percentage",
    ]
]

print(pressure_target_summary)

   pressure      outcome   rows  percentage
0         0  No pressure  32045      88.378
1         1     Pressure   4214      11.622


In [97]:
# Stage 3.7 — Calculate pressure prevalence by actual week

EXPECTED_ACTUAL_WEEKS = set(range(1, 9))

assert not pressure_data["actual_week"].isna().any()

pressure_by_week = (
    pressure_data
    .groupby("actual_week", as_index=False)
    .agg(
        pass_rush_rows=(TARGET_COLUMN, "size"),
        pressure_positives=(TARGET_COLUMN, "sum"),
    )
)

pressure_by_week["pressure_positives"] = (
    pressure_by_week["pressure_positives"]
    .astype("int64")
)

pressure_by_week["pressure_negatives"] = (
    pressure_by_week["pass_rush_rows"]
    - pressure_by_week["pressure_positives"]
)

pressure_by_week["prevalence_pct"] = (
    pressure_by_week["pressure_positives"]
    .div(pressure_by_week["pass_rush_rows"])
    .mul(100)
    .round(3)
)

pressure_by_week = pressure_by_week[
    [
        "actual_week",
        "pass_rush_rows",
        "pressure_negatives",
        "pressure_positives",
        "prevalence_pct",
    ]
]

assert set(pressure_by_week["actual_week"]) == EXPECTED_ACTUAL_WEEKS
assert pressure_by_week["pass_rush_rows"].sum() == 36_259
assert pressure_by_week["pressure_negatives"].sum() == 32_045
assert pressure_by_week["pressure_positives"].sum() == 4_214

print(pressure_by_week)

   actual_week  pass_rush_rows  pressure_negatives  pressure_positives  \
0            1            4988                4390                 598   
1            2            4498                3977                 521   
2            3            4809                4203                 606   
3            4            4733                4176                 557   
4            5            4681                4153                 528   
5            6            4241                3768                 473   
6            7            3865                3438                 427   
7            8            4444                3940                 504   

   prevalence_pct  
0          11.989  
1          11.583  
2          12.601  
3          11.768  
4          11.280  
5          11.153  
6          11.048  
7          11.341  


In [98]:
# Stage 3.8 — Compare pressure before and after snap exclusion

EXPECTED_ALL_PASS_RUSH_ROWS = 36_362
EXPECTED_ALL_PRESSURE_POSITIVES = 4_232
EXPECTED_EXCLUDED_PASS_RUSH_ROWS = 103

all_pass_rush_data = (
    pff_with_snap_quality.loc[
        pff_with_snap_quality["pff_role"].eq(PASS_RUSH_ROLE)
    ]
    .copy()
)

assert (
    all_pass_rush_data[PRESSURE_COMPONENT_COLUMNS]
    .notna()
    .all()
    .all()
)

all_pass_rush_data[TARGET_COLUMN] = (
    all_pass_rush_data[PRESSURE_COMPONENT_COLUMNS]
    .eq(1)
    .any(axis=1)
    .astype("int8")
)

pressure_before_after_snap = pd.DataFrame(
    [
        {
            "universe": "Before snap exclusion",
            "pass_rush_rows": len(all_pass_rush_data),
            "pressure_positives": int(
                all_pass_rush_data[TARGET_COLUMN].sum()
            ),
        },
        {
            "universe": "After snap exclusion",
            "pass_rush_rows": len(pressure_data),
            "pressure_positives": int(
                pressure_data[TARGET_COLUMN].sum()
            ),
        },
    ]
)

pressure_before_after_snap["pressure_negatives"] = (
    pressure_before_after_snap["pass_rush_rows"]
    - pressure_before_after_snap["pressure_positives"]
)

pressure_before_after_snap["prevalence_pct"] = (
    pressure_before_after_snap["pressure_positives"]
    .div(pressure_before_after_snap["pass_rush_rows"])
    .mul(100)
    .round(3)
)

pressure_before_after_snap = pressure_before_after_snap[
    [
        "universe",
        "pass_rush_rows",
        "pressure_negatives",
        "pressure_positives",
        "prevalence_pct",
    ]
]

assert len(all_pass_rush_data) == EXPECTED_ALL_PASS_RUSH_ROWS
assert (
    all_pass_rush_data[TARGET_COLUMN].sum()
    == EXPECTED_ALL_PRESSURE_POSITIVES
)
assert (
    len(all_pass_rush_data) - len(pressure_data)
    == EXPECTED_EXCLUDED_PASS_RUSH_ROWS
)

print(pressure_before_after_snap)

                universe  pass_rush_rows  pressure_negatives  \
0  Before snap exclusion           36362               32130   
1   After snap exclusion           36259               32045   

   pressure_positives  prevalence_pct  
0                4232          11.639  
1                4214          11.622  


### Pressure target definition

La unidad de análisis es un defensor individual clasificado por PFF con el rol
`Pass Rush` dentro de una jugada. Un mismo partido y una misma jugada pueden
generar varias filas, una por cada pass rusher.

La variable objetivo binaria `pressure` indica si el defensor consiguió afectar
al quarterback durante la jugada:

- `pressure = 1`: el defensor produjo al menos un `hurry`, `hit` o `sack`.
- `pressure = 0`: el defensor no produjo ninguno de esos resultados.

Un `hurry` ocurre cuando el defensor obliga al quarterback a apresurar o
modificar su acción. Un `hit` indica que el defensor logró golpearlo. Un `sack`
ocurre cuando lo derriba antes de que complete el pase.

La variable se construyó mediante un OR lógico entre `pff_hurry`, `pff_hit` y
`pff_sack`. Aunque estos eventos podrían superponerse conceptualmente, no se
observaron superposiciones en el universo retenido.

El universo final contiene 36,259 registros Pass Rush pertenecientes a 8,531
jugadas con `ball_snap` manual válido. Se identificaron 4,214 casos positivos
(11.622%) y 32,045 negativos (88.378%).

La exclusión de 103 registros Pass Rush sin un snap manual válido redujo la
prevalencia únicamente de 11.639% a 11.622%, por lo que su impacto agregado
sobre la distribución del objetivo fue mínimo.

### Data leakage prevention

`Data leakage` ocurre cuando un modelo recibe información que no estaría
disponible en el momento real de realizar la predicción. Esto puede producir
métricas artificialmente altas sin que el modelo sea útil en la práctica.

El objetivo del proyecto es predecir presión utilizando únicamente información
disponible en el instante exacto del `ball_snap`. Por tanto, las variables
`pff_hurry`, `pff_hit` y `pff_sack` pueden utilizarse para construir la variable
objetivo, pero nunca pueden utilizarse como predictores.

También quedan excluidos como predictores:

- La propia variable `pressure`.
- Trayectorias, posiciones o velocidades posteriores al snap.
- Eventos registrados después del snap.
- Variables PFF que describan el desarrollo o resultado de la jugada.
- `gameId`, `playId` y `nflId`, que se conservarán únicamente para trazabilidad,
  uniones, agrupamientos y validaciones.
- `actual_week`, que se utilizará para realizar la separación temporal.
- `has_valid_manual_snap`, que se utilizará únicamente como control de calidad.

Los predictores del modelo principal se construirán posteriormente a partir del
frame exacto del snap. Podrán incluir posiciones, distancias, velocidad,
aceleración, orientación, dirección de movimiento y relaciones espaciales entre
el quarterback, los pass rushers, los bloqueadores y el balón.

Antes de entrenar cualquier modelo se validará explícitamente que ninguna
variable objetivo, posterior al snap o de identificación esté presente en la
matriz de predictores.

In [100]:
# Stage 3.11 — Run final pressure target integrity checks

TARGET_KEY_COLUMNS = [
    "gameId",
    "playId",
    "nflId",
]

recomputed_pressure = (
    pressure_data[PRESSURE_COMPONENT_COLUMNS]
    .eq(1)
    .any(axis=1)
    .astype("int8")
)

target_counts = pressure_data[TARGET_COLUMN].value_counts()

target_validation_checks = pd.Series(
    {
        "Expected row count": (
            len(pressure_data)
            == EXPECTED_VALID_PASS_RUSH_ROWS
        ),
        "Complete player-play key": (
            pressure_data[TARGET_KEY_COLUMNS]
            .notna()
            .all()
            .all()
        ),
        "Unique player-play key": (
            not pressure_data.duplicated(
                subset=TARGET_KEY_COLUMNS
            ).any()
        ),
        "Pass Rush role only": (
            pressure_data["pff_role"]
            .eq(PASS_RUSH_ROLE)
            .all()
        ),
        "Valid manual snap only": (
            pressure_data["has_valid_manual_snap"]
            .eq(True)
            .all()
        ),
        "Expected modeling play count": (
            pressure_data[["gameId", "playId"]]
            .drop_duplicates()
            .shape[0]
            == EXPECTED_PASS_RUSH_PLAY_COUNT
        ),
        "Expected actual weeks": (
            set(pressure_data["actual_week"])
            == EXPECTED_ACTUAL_WEEKS
        ),
        "Complete pressure components": (
            pressure_data[PRESSURE_COMPONENT_COLUMNS]
            .notna()
            .all()
            .all()
        ),
        "Complete pressure target": (
            pressure_data[TARGET_COLUMN]
            .notna()
            .all()
        ),
        "Binary pressure target": (
            set(pressure_data[TARGET_COLUMN].unique())
            == {0, 1}
        ),
        "Pressure matches component OR": (
            pressure_data[TARGET_COLUMN]
            .eq(recomputed_pressure)
            .all()
        ),
        "Expected negative count": (
            int(target_counts.get(0, 0))
            == EXPECTED_PRESSURE_NEGATIVES
        ),
        "Expected positive count": (
            int(target_counts.get(1, 0))
            == EXPECTED_PRESSURE_POSITIVES
        ),
    },
    name="passed",
    dtype="bool",
)

failed_target_checks = target_validation_checks.index[
    ~target_validation_checks
].tolist()

assert not failed_target_checks, (
    f"Failed target checks: {failed_target_checks}"
)

target_validation_report = (
    target_validation_checks
    .rename_axis("check")
    .reset_index()
)

target_validation_report["status"] = (
    target_validation_report["passed"]
    .map({True: "PASS", False: "FAIL"})
)

print(target_validation_report[["check", "status"]])

                            check status
0              Expected row count   PASS
1        Complete player-play key   PASS
2          Unique player-play key   PASS
3             Pass Rush role only   PASS
4          Valid manual snap only   PASS
5    Expected modeling play count   PASS
6           Expected actual weeks   PASS
7    Complete pressure components   PASS
8        Complete pressure target   PASS
9          Binary pressure target   PASS
10  Pressure matches component OR   PASS
11        Expected negative count   PASS
12        Expected positive count   PASS
